# LeetCode 4060 — Count Evenly Good Integers (Hard)

Notebook này dạy bài **Count Evenly Good Integers**: đếm các số nguyên trong đoạn `[l, r]` có **số lượng chữ số chẵn là một số chẵn**.

```python
class Solution:
    def countEvenlyGoodIntegers(self, l: int, r: int) -> int:
        ...
```

Bài thuộc nhóm **Digit DP** (quy hoạch động theo chữ số) và **toán đếm** (combinatorics). Đây là kiểu bài "đếm số trong một đoạn khổng lồ": `r` có thể tới `10^15`, nên ta **không thể** kiểm tra từng số. Ta sẽ đi từ cách ngây thơ nhất tới một công thức chỉ vài dòng, và hiểu vì sao từng bước đúng.

---

## Sau notebook này, bạn sẽ nắm được

1. Đọc đúng đề: phân biệt **chữ số chẵn** với **số chẵn**, và vì sao chữ số `0` cũng được tính là chữ số chẵn.
2. Kỹ thuật **đếm theo tiền tố**: `đếm trong [l, r] = F(r) - F(l - 1)`.
3. Vì sao chỉ cần nhớ **parity** (tính chẵn/lẻ) của số chữ số chẵn, không cần nhớ chính xác là bao nhiêu.
4. Khung **Digit DP** với trạng thái `(pos, parity, tight, started)`: `tight` là gì, vì sao phải xử lý **leading zero** (số 0 đứng đầu).
5. Một định luật đếm đẹp: nếu còn `m >= 1` chữ số tự do thì **đúng một nửa** trong `10^m` cách điền làm số chữ số chẵn là chẵn.
6. Quan sát "mỗi khối 10 số `[10q, 10q + 9]` có đúng 5 số tốt" và công thức `F(x) = 5 * (x // 10) + phần đuôi`.
7. Kiểm thử đối chiếu nhiều lời giải với brute force, và nhận ra dạng bài digit DP ở những bài khác.

## Các cách giải sẽ học (từ dễ đến khéo)

| Cách | Ý tưởng chính | Thời gian |
|---|---|---|
| Cách 1 | Brute force: duyệt từng số, đếm chữ số chẵn | `O((r - l + 1) * D)` |
| Cách 2 | Digit DP có memo: điền chữ số từ trái sang phải | `O(D * 8 * 10)` |
| Cách 3 | Toán đếm: đếm theo độ dài + đi dọc các chữ số của `x` | `O(D * 10)` |
| Cách 4 | Khối 10 số: `F(x) = 5 * (x // 10) + đuôi` | `O(D)` |

Trong đó `D` là số chữ số của `r` (với `r <= 10^15` thì `D <= 16`).

# 1. Đề bài (dịch tiếng Việt)

Cho hai số nguyên `l` và `r`.

Một số nguyên được gọi là **evenly good** (tạm dịch: "tốt đều") nếu nó chứa **một số chẵn các chữ số chẵn** — nghĩa là: đếm xem trong cách viết thập phân của số đó có bao nhiêu chữ số thuộc `{0, 2, 4, 6, 8}`, nếu số lượng này là `0, 2, 4, ...` thì số đó là evenly good.

Hãy trả về **số lượng** số nguyên evenly good trong đoạn **bao gồm cả hai đầu** `[l, r]`.

Skeleton trên LeetCode:

```python
class Solution:
    def countEvenlyGoodIntegers(self, l: int, r: int) -> int:
```

---

## Ví dụ 1

```text
Input:  l = 18, r = 22
Output: 3
```

Các số evenly good trong `[18, 22]`:
- `19`: có 0 chữ số chẵn.
- `20`: có 2 chữ số chẵn (`2` và `0`).
- `22`: có 2 chữ số chẵn (`2` và `2`).

## Ví dụ 2

```text
Input:  l = 98, r = 101
Output: 2
```

- `99`: có 0 chữ số chẵn.
- `100`: có 2 chữ số chẵn (hai chữ số `0`).

## Ví dụ 3

```text
Input:  l = 1, r = 10
Output: 5
```

Các số `1, 3, 5, 7, 9` đều có 0 chữ số chẵn.

## Ràng buộc

```text
1 <= l <= r <= 10^15
```

---

## Ba điều phải đọc thật kỹ

1. **`0` là một số chẵn**, nên số không có chữ số chẵn nào (như `19`, `99`, `7`) **vẫn là evenly good**.
2. **Chữ số `0` là chữ số chẵn.** `100` có hai chữ số `0` → 2 chữ số chẵn → tốt.
3. Ta xét **chữ số**, không xét bản thân số chẵn hay lẻ: `20` là số chẵn và tốt, `21` là số lẻ nhưng **không** tốt (chỉ có 1 chữ số chẵn là `2`).

In [1]:
# Dữ liệu các ví dụ trong đề: (l, r, đáp án mong đợi)
examples = [
    (18, 22, 3),
    (98, 101, 2),
    (1, 10, 5),
]

for l, r, expected in examples:
    numbers = list(range(l, r + 1))
    print(f"l = {l:>3}, r = {r:>3} | đáp án mong đợi = {expected} | các số trong đoạn: {numbers}")

l =  18, r =  22 | đáp án mong đợi = 3 | các số trong đoạn: [18, 19, 20, 21, 22]
l =  98, r = 101 | đáp án mong đợi = 2 | các số trong đoạn: [98, 99, 100, 101]
l =   1, r =  10 | đáp án mong đợi = 5 | các số trong đoạn: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


## Ràng buộc nói gì về độ phức tạp?

```text
1 <= l <= r <= 10^15
```

- Đoạn `[l, r]` có thể chứa tới khoảng `10^15` số. Python đơn thuần làm được cỡ `10^7` phép tính đơn giản mỗi giây → duyệt hết cần **ít nhất** khoảng `10^8` giây, tức là **hơn 3 năm** (thực tế còn lâu hơn, vì mỗi số tốn nhiều phép). ⇒ Không thể kiểm tra từng số.
- Nhưng mỗi số chỉ có **tối đa 16 chữ số** (`10^15 = 1` theo sau là 15 số `0`). ⇒ Nếu thuật toán làm việc **theo từng chữ số** (vài chục, vài trăm, vài nghìn phép) thì chạy gần như tức thì.
- Đáp án có thể lớn cỡ `5 * 10^14`, vượt giới hạn `int` 32 bit. Trong C++/Java phải dùng `long long`/`long`; Python thì số nguyên không bị tràn.

> Kết luận: cần thuật toán **"đếm theo chữ số"** (`O(D)` hoặc `O(D * hằng số)`), không phải **"đếm theo từng số"**.

# 2. Hiểu đề bằng tay trước khi nghĩ thuật toán

## Ví dụ 1: đoạn `[18, 22]`

| x | các chữ số | chữ số chẵn | số lượng | evenly good? |
|---:|---|---|---:|:---:|
| 18 | 1, 8 | 8 | 1 | Không |
| 19 | 1, 9 | (không có) | 0 | **Có** |
| 20 | 2, 0 | 2, 0 | 2 | **Có** |
| 21 | 2, 1 | 2 | 1 | Không |
| 22 | 2, 2 | 2, 2 | 2 | **Có** |

→ Có 3 số tốt: `19, 20, 22`.

## Ví dụ 2: đoạn `[98, 101]`

| x | các chữ số | chữ số chẵn | số lượng | evenly good? |
|---:|---|---|---:|:---:|
| 98 | 9, 8 | 8 | 1 | Không |
| 99 | 9, 9 | (không có) | 0 | **Có** |
| 100 | 1, 0, 0 | 0, 0 | 2 | **Có** |
| 101 | 1, 0, 1 | 0 | 1 | Không |

→ Có 2 số tốt: `99, 100`.

Để ý cột "số lượng": mỗi lần một chữ số chẵn **xuất hiện** đều được đếm (hai chữ số `2` trong `22` được đếm 2 lần).

Cell dưới đây kiểm tra lại bảng tay bằng code. Ở đây ta dùng `str(x)` để lấy các chữ số và `ch in "02468"` để hỏi "ký tự này có phải chữ số chẵn không" — phần Nền tảng sẽ giải thích kỹ hai công cụ này.

In [2]:
def even_digits_of(x: int) -> list:
    """Trả về danh sách các chữ số chẵn của x (giữ cả các lần lặp lại)."""
    return [ch for ch in str(x) if ch in "02468"]


for l, r in [(18, 22), (98, 101)]:
    print(f"Đoạn [{l}, {r}]")
    good = []
    for x in range(l, r + 1):
        evens = even_digits_of(x)
        ok = len(evens) % 2 == 0
        if ok:
            good.append(x)
        print(f"  x = {x:>3} | chữ số chẵn = {evens} | số lượng = {len(evens)} | tốt: {'Có' if ok else 'Không'}")
    print("  => các số tốt:", good, "| đếm được:", len(good))

Đoạn [18, 22]
  x =  18 | chữ số chẵn = ['8'] | số lượng = 1 | tốt: Không
  x =  19 | chữ số chẵn = [] | số lượng = 0 | tốt: Có
  x =  20 | chữ số chẵn = ['2', '0'] | số lượng = 2 | tốt: Có
  x =  21 | chữ số chẵn = ['2'] | số lượng = 1 | tốt: Không
  x =  22 | chữ số chẵn = ['2', '2'] | số lượng = 2 | tốt: Có
  => các số tốt: [19, 20, 22] | đếm được: 3
Đoạn [98, 101]
  x =  98 | chữ số chẵn = ['8'] | số lượng = 1 | tốt: Không
  x =  99 | chữ số chẵn = [] | số lượng = 0 | tốt: Có
  x = 100 | chữ số chẵn = ['0', '0'] | số lượng = 2 | tốt: Có
  x = 101 | chữ số chẵn = ['0'] | số lượng = 1 | tốt: Không
  => các số tốt: [99, 100] | đếm được: 2


## Thí nghiệm tay nhỏ: đếm theo từng "chục"

Trước khi học thuật toán, hãy tự tính số lượng số tốt trong mỗi nhóm 10 số liên tiếp:

- `[1..9]` (số `0` không thuộc đề vì `l >= 1`): các số lẻ `1, 3, 5, 7, 9` → **5** số.
- `[10..19]`: chữ số đầu `1` là lẻ, nên cần chữ số cuối lẻ: `11, 13, 15, 17, 19` → **5** số.
- `[20..29]`: chữ số đầu `2` đã là 1 chữ số chẵn, cần thêm 1 nữa → chữ số cuối chẵn: `20, 22, 24, 26, 28` → **5** số.

Có vẻ **mỗi chục đều có đúng 5 số tốt**? Cell dưới kiểm tra thêm vài nhóm. Hãy giữ câu hỏi "vì sao?" trong đầu — nó sẽ trở thành **Cách 4**.

In [3]:
def is_good_simple(x: int) -> bool:
    return len(even_digits_of(x)) % 2 == 0


for start in [0, 10, 20, 100, 250]:
    block = range(start, start + 10)
    good = [x for x in block if x >= 1 and is_good_simple(x)]
    print(f"[{start}..{start + 9}] -> {len(good)} số tốt: {good}")

[0..9] -> 5 số tốt: [1, 3, 5, 7, 9]
[10..19] -> 5 số tốt: [11, 13, 15, 17, 19]
[20..29] -> 5 số tốt: [20, 22, 24, 26, 28]
[100..109] -> 5 số tốt: [100, 102, 104, 106, 108]
[250..259] -> 5 số tốt: [250, 252, 254, 256, 258]


# 3. Nền tảng cần biết

Các lời giải phía sau dùng những công cụ sau. Ta học từng cái với ví dụ nhỏ trước, rồi mới ghép vào bài.

| # | Công cụ | Dùng ở đâu |
|---|---|---|
| N1 | Tách chữ số bằng chuỗi: `str(x)`, `int(ch)`, `ch in "02468"` | Brute force, Digit DP |
| N2 | Tách chữ số bằng toán: `% 10`, `// 10`, `divmod` | Cách 4 |
| N3 | Parity (chẵn/lẻ) và phép XOR `^` để "lật" trạng thái | Mọi cách nhanh |
| N4 | Đếm theo tiền tố: `F(r) - F(l - 1)` | Cách 2, 3, 4 |
| N5 | Đệ quy + memoization với `functools.cache` | Cách 2 |
| N6 | Quy tắc nhân, `math.comb`, đếm chuỗi chữ số | Cách 3 |
| N7 | "Bám sát `x`" (tight): liệt kê các số `<= x` theo từng chữ số | Cách 2, 3 |

## N1. Tách chữ số bằng chuỗi

`str(x)` biến số thành chuỗi ký tự. Duyệt chuỗi bằng `for ch in s` cho ta từng chữ số (dạng ký tự).

```python
s = str(2024)        # "2024"
for ch in s:         # "2", "0", "2", "4"
    ...
```

- `int(ch)` đổi ký tự `"7"` thành số `7`.
- `ch in "02468"` hỏi: ký tự `ch` có nằm trong chuỗi `"02468"` không? → đúng khi `ch` là chữ số chẵn.
- `sum(1 for ch in s if ...)` là **generator expression**: đếm có bao nhiêu `ch` thỏa điều kiện (mỗi `ch` thỏa đóng góp `1`).

Liên hệ với bài: đây là cách dễ nhất để viết hàm `even_digit_count(x)` cho brute force.

In [4]:
s = str(2024)
print("s =", repr(s), "| len =", len(s))

for ch in s:
    print(f"ký tự {ch!r} -> số {int(ch)} | là chữ số chẵn? {ch in '02468'}")


def even_digit_count(x: int) -> int:
    """Đếm số chữ số chẵn trong x (dùng chuỗi)."""
    return sum(1 for ch in str(x) if ch in "02468")


print("even_digit_count(2024) =", even_digit_count(2024))
print("even_digit_count(13579) =", even_digit_count(13579))
print("even_digit_count(100) =", even_digit_count(100))

s = '2024' | len = 4
ký tự '2' -> số 2 | là chữ số chẵn? True
ký tự '0' -> số 0 | là chữ số chẵn? True
ký tự '2' -> số 2 | là chữ số chẵn? True
ký tự '4' -> số 4 | là chữ số chẵn? True
even_digit_count(2024) = 4
even_digit_count(13579) = 0
even_digit_count(100) = 2


## N2. Tách chữ số bằng toán: `% 10`, `// 10`, `divmod`

Với số nguyên dương `x`:

```python
x % 10    # chữ số cuối cùng (hàng đơn vị)
x // 10   # bỏ chữ số cuối cùng (chia lấy phần nguyên)
```

Ví dụ `x = 253`: `253 % 10 = 3`, `253 // 10 = 25`.

`divmod(x, 10)` trả về **cả hai** cùng lúc: `divmod(253, 10) = (25, 3)`.

Lặp lại "lấy chữ số cuối, rồi bỏ nó" cho tới khi `x == 0` là ta đi qua mọi chữ số (theo thứ tự từ phải sang trái).

Liên hệ với bài: ở **Cách 4**, ta tách `x = 10 * q + t` bằng `q, t = divmod(x, 10)`: `q` là "phần đầu", `t` là chữ số cuối.

In [5]:
x = 253
print("253 % 10  =", x % 10)
print("253 // 10 =", x // 10)
print("divmod(253, 10) =", divmod(x, 10))

# Đi qua các chữ số của 2024 bằng toán (từ phải sang trái)
y = 2024
digits_right_to_left = []
while y > 0:
    y, last = divmod(y, 10)
    digits_right_to_left.append(last)
print("Các chữ số của 2024 (phải -> trái):", digits_right_to_left)

253 % 10  = 3
253 // 10 = 25
divmod(253, 10) = (25, 3)
Các chữ số của 2024 (phải -> trái): [4, 2, 0, 2]


## N3. Parity (chẵn/lẻ) và phép XOR để "lật"

**Parity** của một số nguyên là nó chẵn (`0`) hay lẻ (`1`), tức là `n % 2`.

- Chữ số `d` là chẵn ⇔ `d % 2 == 0` (cũng có thể viết `d & 1 == 0`: bit cuối bằng 0).
- Ta không cần biết "đã gặp **bao nhiêu** chữ số chẵn", chỉ cần biết số đó **chẵn hay lẻ**. Mỗi lần gặp một chữ số chẵn, parity **lật**: `0 → 1`, `1 → 0`.
- Phép **XOR** `^` với `1` chính là lật: `0 ^ 1 = 1`, `1 ^ 1 = 0`; còn `p ^ 0 = p` (giữ nguyên).

Ví dụ `2024`:

```text
ban đầu parity = 0
gặp 2 (chẵn): parity = 0 ^ 1 = 1
gặp 0 (chẵn): parity = 1 ^ 1 = 0
gặp 2 (chẵn): parity = 0 ^ 1 = 1
gặp 4 (chẵn): parity = 1 ^ 1 = 0   -> 4 chữ số chẵn, parity 0 -> TỐT
```

Liên hệ với bài: "số tốt" ⇔ **parity cuối cùng bằng 0**. Đây là trạng thái chỉ có 2 giá trị — chìa khóa để các lời giải nhanh có ít trạng thái.

In [6]:
print("0 ^ 1 =", 0 ^ 1, "| 1 ^ 1 =", 1 ^ 1, "| 0 ^ 0 =", 0 ^ 0, "| 1 ^ 0 =", 1 ^ 0)
print("7 % 2 =", 7 % 2, "| 7 & 1 =", 7 & 1, "| 8 % 2 =", 8 % 2, "| 8 & 1 =", 8 & 1)

parity = 0
for ch in "2024":
    d = int(ch)
    if d % 2 == 0:
        parity ^= 1          # gặp chữ số chẵn -> lật parity
    print(f"gặp {d}: parity = {parity}")
print("parity cuối = 0 -> 2024 là số tốt?", parity == 0)

0 ^ 1 = 1 | 1 ^ 1 = 0 | 0 ^ 0 = 0 | 1 ^ 0 = 1
7 % 2 = 1 | 7 & 1 = 1 | 8 % 2 = 0 | 8 & 1 = 0
gặp 2: parity = 1
gặp 0: parity = 0
gặp 2: parity = 1
gặp 4: parity = 0
parity cuối = 0 -> 2024 là số tốt? True


## N4. Đếm trong đoạn bằng hiệu hai tiền tố: `F(r) - F(l - 1)`

Gọi `F(x)` = số lượng số tốt trong `[1, x]`. Khi đó:

```text
đếm trong [l, r] = F(r) - F(l - 1)
```

Vì sao `l - 1` chứ không phải `l`? `F(r)` đếm `1..r`, `F(l - 1)` đếm `1..l-1`. Trừ đi còn đúng `l..r` — **giữ lại** `l`. Nếu trừ `F(l)` thì ta lỡ bỏ mất chính số `l`.

Ví dụ dễ kiểm tra với một tính chất khác: đếm số chia hết cho 3 trong `[10, 20]`.
- `G(x) = x // 3` = số bội của 3 trong `[1, x]`.
- `G(20) - G(9) = 6 - 3 = 3` → đúng là `12, 15, 18`.

Đây cũng chính là ý tưởng của **prefix sum** (tổng tiền tố) trên mảng.

Liên hệ với bài: bài toán hai đầu `[l, r]` trở thành bài toán **một đầu** "đếm trong `[1, x]`" — dễ nghĩ hơn nhiều.

In [7]:
def G(x: int) -> int:
    """Số bội của 3 trong [1, x]."""
    return x // 3

l, r = 10, 20
print("G(20) - G(9)  =", G(r) - G(l - 1), "->", [x for x in range(l, r + 1) if x % 3 == 0])
print("G(20) - G(10) =", G(r) - G(l), "-> tình cờ đúng vì 10 không chia hết cho 3")
print(f"Với l = 12: G(20) - G(12) = {G(20) - G(12)} (SAI, mất số 12), G(20) - G(11) = {G(20) - G(11)} (ĐÚNG)")

G(20) - G(9)  = 3 -> [12, 15, 18]
G(20) - G(10) = 3 -> tình cờ đúng vì 10 không chia hết cho 3
Với l = 12: G(20) - G(12) = 2 (SAI, mất số 12), G(20) - G(11) = 3 (ĐÚNG)


## N5. Đệ quy + memoization với `functools.cache`

**Đệ quy** (recursion): hàm tự gọi lại chính nó với bài toán nhỏ hơn, có **điểm dừng** (base case).

**Memoization**: nhớ kết quả của các lời gọi đã tính để lần sau gặp lại cùng tham số thì trả ngay. Trong Python chỉ cần đặt `@cache` (từ `functools`) phía trên hàm. `f.cache_info()` cho biết số lần `hits` (lấy từ bộ nhớ) và `misses` (phải tính mới).

Ví dụ rất gần bài của ta: `fill(m, parity)` = số cách điền **`m` chữ số bất kỳ (0–9)** sao cho, tính cả `parity` hiện có, tổng số chữ số chẵn là chẵn.

```text
fill(0, parity) = 1 nếu parity == 0, ngược lại 0     (hết chỗ điền: kiểm tra kết quả)
fill(m, parity) = tổng theo d = 0..9 của fill(m - 1, parity mới)
```

Không memo: mỗi tầng gọi 10 lần → `1 + 10 + 100 + 1000 + 10000 = 11111` lời gọi cho `m = 4`. Có memo: chỉ có vài trạng thái `(m, parity)` khác nhau, nên chỉ tính vài lần.

In [8]:
from functools import cache

calls = 0

def fill_plain(m: int, parity: int) -> int:
    global calls
    calls += 1
    if m == 0:
        return 1 if parity == 0 else 0
    total = 0
    for d in range(10):
        new_parity = parity ^ 1 if d % 2 == 0 else parity
        total += fill_plain(m - 1, new_parity)
    return total


@cache
def fill_memo(m: int, parity: int) -> int:
    if m == 0:
        return 1 if parity == 0 else 0
    total = 0
    for d in range(10):
        new_parity = parity ^ 1 if d % 2 == 0 else parity
        total += fill_memo(m - 1, new_parity)
    return total


print("fill_plain(4, 0) =", fill_plain(4, 0), "| số lời gọi:", calls)
print("fill_memo(4, 0)  =", fill_memo(4, 0), "|", fill_memo.cache_info())

fill_plain(4, 0) = 5000 | số lời gọi: 11111
fill_memo(4, 0)  = 5000 | CacheInfo(hits=62, misses=9, maxsize=None, currsize=9)


## N6. Quy tắc nhân và `math.comb`: đếm các chuỗi chữ số

- Có `10` chữ số `0..9`: **5 chẵn** (`0, 2, 4, 6, 8`) và **5 lẻ** (`1, 3, 5, 7, 9`).
- **Quy tắc nhân**: điền `m` vị trí độc lập, mỗi vị trí 10 cách → `10^m` chuỗi.
- Số chuỗi dài `m` có **đúng `j`** chữ số chẵn: chọn `j` vị trí cho chữ số chẵn (`C(m, j)` cách — tổ hợp chập `j` của `m`), mỗi vị trí chẵn có 5 lựa chọn, mỗi vị trí lẻ có 5 lựa chọn:

```text
số chuỗi = C(m, j) * 5^j * 5^(m - j) = C(m, j) * 5^m
```

Trong Python: `math.comb(m, j)` tính `C(m, j)`, `5 ** m` tính lũy thừa.

Ví dụ `m = 2`: `j = 0` → 25 chuỗi, `j = 1` → 50 chuỗi, `j = 2` → 25 chuỗi. Số chuỗi có **số chẵn** chữ số chẵn (`j = 0` hoặc `j = 2`) là `25 + 25 = 50` = **đúng một nửa** của 100. Ghi nhớ con số "một nửa" này!

Liên hệ với bài: **Cách 3** đếm "phần đuôi tự do" bằng công thức thay vì liệt kê.

In [9]:
from math import comb

m = 2
total_even_j = 0
for j in range(m + 1):
    ways = comb(m, j) * 5 ** j * 5 ** (m - j)
    print(f"m = {m}, đúng j = {j} chữ số chẵn: C({m},{j}) * 5^{j} * 5^{m - j} = {ways}")
    if j % 2 == 0:
        total_even_j += ways
print("Tổng các j chẵn:", total_even_j, "trên tổng số", 10 ** m, "chuỗi")

# Kiểm tra bằng liệt kê thật: mọi chuỗi 2 chữ số "00".."99"
listed = sum(1 for v in range(100) if sum(1 for ch in f"{v:02d}" if ch in "02468") % 2 == 0)
print("Liệt kê trực tiếp:", listed)

m = 2, đúng j = 0 chữ số chẵn: C(2,0) * 5^0 * 5^2 = 25
m = 2, đúng j = 1 chữ số chẵn: C(2,1) * 5^1 * 5^1 = 50
m = 2, đúng j = 2 chữ số chẵn: C(2,2) * 5^2 * 5^0 = 25
Tổng các j chẵn: 50 trên tổng số 100 chuỗi
Liệt kê trực tiếp: 50


## N7. "Bám sát `x`" (tight): liệt kê các số `<= x` theo từng chữ số

Hai số **cùng số chữ số** được so sánh như so sánh chuỗi: nhìn từ trái sang, **vị trí đầu tiên khác nhau** quyết định số nào lớn hơn.

Vậy mọi số 3 chữ số `y <= x = 253` thuộc đúng **một** trong các nhóm sau (theo vị trí đầu tiên mà `y` nhỏ hơn `x`):

| Nhóm | Mô tả | Các số | Số lượng |
|---|---|---|---:|
| 1 | chữ số đầu `< 2` (không được là 0) | `100..199` | 100 |
| 2 | chữ số đầu `= 2`, chữ số thứ hai `< 5` | `200..249` | 50 |
| 3 | `25` rồi chữ số cuối `< 3` | `250..252` | 3 |
| 4 | trùng hoàn toàn với `x` | `253` | 1 |

Ý tưởng quan trọng: **ngay khi `y` đã nhỏ hơn `x` tại một vị trí, mọi chữ số phía sau được tự do `0..9`**. Còn khi `y` vẫn "bám sát" (bằng) `x` thì chữ số tiếp theo bị giới hạn bởi chữ số của `x`. Trạng thái "đang bám sát hay không" gọi là **`tight`**.

Liên hệ với bài: Cách 2 (digit DP) và Cách 3 (toán đếm) đều đi theo đúng cách phân nhóm này.

In [10]:
x = 253
groups = [
    ("chữ số đầu < 2", range(100, 200)),
    ("'2' rồi chữ số thứ hai < 5", range(200, 250)),
    ("'25' rồi chữ số cuối < 3", range(250, 253)),
    ("chính x = 253", range(253, 254)),
]
total = 0
for name, rng in groups:
    total += len(rng)
    print(f"{name:<28} -> {rng.start}..{rng.stop - 1}: {len(rng)} số")
print("Tổng:", total, "| kiểm tra: số các số 3 chữ số <= 253 =", 253 - 100 + 1)

chữ số đầu < 2               -> 100..199: 100 số
'2' rồi chữ số thứ hai < 5   -> 200..249: 50 số
'25' rồi chữ số cuối < 3     -> 250..252: 3 số
chính x = 253                -> 253..253: 1 số
Tổng: 154 | kiểm tra: số các số 3 chữ số <= 253 = 154


# 4. Cách 1 — Brute force: duyệt từng số

Cách tự nhiên nhất, làm **đúng y như đề nói**:

1. Duyệt `x` từ `l` tới `r`.
2. Đếm số chữ số chẵn của `x`.
3. Nếu số lượng đó chẵn thì tăng biến đếm.

Cách này chắc chắn đúng (nó chỉ là định nghĩa viết thành code), nên ta dùng nó làm **"trọng tài"** để kiểm tra các cách nhanh hơn ở phía sau.

In [11]:
def is_evenly_good(x: int) -> bool:
    return even_digit_count(x) % 2 == 0


def count_evenly_good_bruteforce(l: int, r: int) -> int:
    count = 0
    for x in range(l, r + 1):
        if is_evenly_good(x):
            count += 1
    return count


for l, r, expected in examples:
    got = count_evenly_good_bruteforce(l, r)
    print(f"[{l}, {r}] -> {got} | mong đợi {expected} | {'PASS' if got == expected else 'FAIL'}")
    assert got == expected

[18, 22] -> 3 | mong đợi 3 | PASS
[98, 101] -> 2 | mong đợi 2 | PASS
[1, 10] -> 5 | mong đợi 5 | PASS


## Độ phức tạp của brute force và vì sao quá chậm

Gọi `N = r - l + 1` (số lượng số cần xét) và `D` = số chữ số (tối đa 16).

- Mỗi số tốn `O(D)` để đếm chữ số chẵn.
- Tổng thời gian: `O(N * D)`.
- Bộ nhớ: `O(D)` (chuỗi tạm `str(x)`).

Với `N` tới `10^15`, đây là con số không tưởng. Cell dưới đo thời gian thật trên `200 000` số rồi **ước lượng** cho `10^15` số (con số cụ thể tùy máy, nhưng luôn là **nhiều năm**).

Nút thắt: ta **lặp lại công việc giống nhau** vô số lần. Ví dụ 100 triệu số từ `500 000 000 000 000` tới `500 000 099 999 999` có chung phần đầu `5000000` — brute force đếm lại chữ số chẵn của phần đầu đó **100 triệu lần**. Các cách sau sẽ **gom nhóm** những số có cùng phần đầu.

In [12]:
import time

n_test = 200_000
start = time.perf_counter()
count_evenly_good_bruteforce(1, n_test)
elapsed = time.perf_counter() - start

per_number = elapsed / n_test
estimate = 10 ** 15 * per_number
print(f"{n_test} số mất khoảng {elapsed:.3f} giây")
print(f"Ước lượng cho 10^15 số: {estimate:.2e} giây ≈ {estimate / 3600 / 24 / 365:,.0f} năm")

200000 số mất khoảng 0.100 giây
Ước lượng cho 10^15 số: 5.01e+08 giây ≈ 16 năm


# 5. Bước đột phá 1 — Chuyển "đoạn" thành "tiền tố"

Dùng nền tảng N4: định nghĩa

```text
F(x) = số lượng số evenly good trong [1, x]       (F(0) = 0)
đáp án = F(r) - F(l - 1)
```

Ví dụ 1: `F(22) - F(17)`. Ví dụ 3: `F(10) - F(0)`.

Từ giờ ta chỉ cần giải bài **một đầu**: "có bao nhiêu số tốt từ 1 tới `x`?". Cell dưới kiểm tra công thức bằng một `F` brute force.

In [13]:
def F_bruteforce(x: int) -> int:
    return count_evenly_good_bruteforce(1, x) if x >= 1 else 0


for l, r, expected in examples:
    a, b = F_bruteforce(r), F_bruteforce(l - 1)
    print(f"F({r}) - F({l - 1}) = {a} - {b} = {a - b} | mong đợi {expected}")
    assert a - b == expected

F(22) - F(17) = 12 - 9 = 3 | mong đợi 3
F(101) - F(97) = 51 - 49 = 2 | mong đợi 2
F(10) - F(0) = 5 - 0 = 5 | mong đợi 5


# 6. Bước đột phá 2 — Chỉ cần nhớ parity của phần đầu

Hãy tưởng tượng ta **viết số từ trái sang phải**. Khi đã viết xong một phần đầu (prefix), điều duy nhất ảnh hưởng tới kết luận "tốt hay không" là **parity** số chữ số chẵn của prefix đó.

Ví dụ prefix `"20"` (2 chữ số chẵn → parity 0) và prefix `"13"` (0 chữ số chẵn → parity 0): với **bất kỳ** phần đuôi nào ghép vào, hai số tạo ra luôn cùng tốt hoặc cùng không tốt. Ngược lại prefix `"21"` (parity 1) cho kết quả **ngược lại**.

Hệ quả: dù prefix có thể là hàng tỷ chuỗi khác nhau, về mặt bài toán chỉ có **2 loại** prefix: parity 0 và parity 1. Đây là lý do digit DP ở Cách 2 chỉ cần vài chục trạng thái.

In [14]:
suffixes = ["0", "1", "7", "8", "00", "35", "246"]
for suf in suffixes:
    a = is_evenly_good(int("20" + suf))
    b = is_evenly_good(int("13" + suf))
    c = is_evenly_good(int("21" + suf))
    print(f"đuôi {suf:>4}: '20'+đuôi tốt? {a!s:<5} | '13'+đuôi tốt? {b!s:<5} | '21'+đuôi tốt? {c}")
    assert a == b and a != c
print("=> prefix cùng parity cho kết quả như nhau; khác parity cho kết quả ngược nhau.")

đuôi    0: '20'+đuôi tốt? False | '13'+đuôi tốt? False | '21'+đuôi tốt? True
đuôi    1: '20'+đuôi tốt? True  | '13'+đuôi tốt? True  | '21'+đuôi tốt? False
đuôi    7: '20'+đuôi tốt? True  | '13'+đuôi tốt? True  | '21'+đuôi tốt? False
đuôi    8: '20'+đuôi tốt? False | '13'+đuôi tốt? False | '21'+đuôi tốt? True
đuôi   00: '20'+đuôi tốt? True  | '13'+đuôi tốt? True  | '21'+đuôi tốt? False
đuôi   35: '20'+đuôi tốt? True  | '13'+đuôi tốt? True  | '21'+đuôi tốt? False
đuôi  246: '20'+đuôi tốt? False | '13'+đuôi tốt? False | '21'+đuôi tốt? True
=> prefix cùng parity cho kết quả như nhau; khác parity cho kết quả ngược nhau.


# 7. Cách 2 — Digit DP (quy hoạch động theo chữ số)

## Ý tưởng

Ta đếm `F(x)` bằng cách **dựng từng số `y <= x` một chữ số mỗi bước**, từ trái sang phải, theo đúng khung "bám sát `x`" của N7. Gọi `s = str(x)` và `n = len(s)`.

Mọi số `y` trong `[0, x]` được viết thành chuỗi **đúng `n` ký tự** bằng cách thêm số 0 ở đầu. Ví dụ với `x = 22`: số `5` được viết là `"05"`.

## Trạng thái `dfs(pos, parity, tight, started)`

| Tham số | Ý nghĩa |
|---|---|
| `pos` | đang chọn chữ số ở vị trí thứ `pos` (0 = trái nhất) |
| `parity` | parity số chữ số chẵn **thật** đã đặt (0 chẵn, 1 lẻ) |
| `tight` | `True` nếu các chữ số đã đặt **bằng đúng** prefix của `x` → chữ số tiếp theo không được vượt `s[pos]` |
| `started` | `True` nếu đã đặt ít nhất một chữ số khác 0 → số đã "bắt đầu"; các số 0 trước đó chỉ là số 0 đệm |

`dfs(...)` trả về: **số cách điền các vị trí `pos..n-1`** để được một số tốt (và `<= x`).

## Chuyển trạng thái

- Giới hạn chữ số: `up = int(s[pos])` nếu `tight`, ngược lại `up = 9`.
- Thử mọi `d` từ `0` tới `up`:
  - `new_started = started or d != 0`
  - nếu `new_started` và `d` chẵn → lật parity (số 0 đệm **không** được tính).
  - `new_tight = tight and d == up` (chỉ còn bám sát nếu ta chọn đúng chữ số của `x`).
- **Base case** `pos == n`: trả `1` nếu `started` (số khác 0 — đề chỉ xét số `>= 1`) **và** `parity == 0`; ngược lại trả `0`.

## Vì sao cần `started` (xử lý leading zero)?

Với `x = 22`, số `5` được dựng thành `"05"`. Nếu ta coi chữ số `0` đệm đó là chữ số chẵn thật, số `5` sẽ bị tính là "1 chữ số chẵn" → không tốt — **sai**, vì `5` thật ra có 0 chữ số chẵn.

Cờ `started` nói: "chừng nào chưa gặp chữ số khác 0 thì các số 0 chỉ là đệm, bỏ qua". Đây là thói quen chuẩn của digit DP: **luôn tự hỏi số 0 đứng đầu có ảnh hưởng tới tính chất đang đếm không**.

> Điều thú vị: riêng bài này, nếu quên `started` thì **tổng số đếm vẫn tình cờ đúng** (dù từng số cụ thể bị phân loại sai). Ở phần "Lỗi thường gặp" ta sẽ xem vì sao, và một biến thể mà thiếu `started` thì sai be bét.

## Một chi tiết Python: `@cache` đặt **bên trong** hàm

Ta định nghĩa `dfs` bên trong `count_upto_digit_dp(x)`. Mỗi lần gọi `count_upto_digit_dp` sẽ tạo ra một `dfs` mới với **bộ nhớ cache mới**, nên `F(r)` và `F(l - 1)` không bao giờ dùng nhầm kết quả của nhau (lỗi này có thật — xem phần Lỗi thường gặp).

In [15]:
from functools import cache


def count_upto_digit_dp(x: int) -> int:
    """F(x): số lượng số evenly good trong [1, x], bằng digit DP."""
    if x <= 0:
        return 0
    s = str(x)
    n = len(s)

    @cache
    def dfs(pos: int, parity: int, tight: bool, started: bool) -> int:
        if pos == n:
            return 1 if (started and parity == 0) else 0
        up = int(s[pos]) if tight else 9
        total = 0
        for d in range(up + 1):
            new_started = started or d != 0
            new_parity = parity
            if new_started and d % 2 == 0:
                new_parity ^= 1                  # chữ số chẵn THẬT -> lật parity
            total += dfs(pos + 1, new_parity, tight and d == up, new_started)
        return total

    result = dfs(0, 0, True, False)
    count_upto_digit_dp.last_cache_info = dfs.cache_info()   # chỉ để quan sát
    return result


def count_evenly_good_digit_dp(l: int, r: int) -> int:
    return count_upto_digit_dp(r) - count_upto_digit_dp(l - 1)


for l, r, expected in examples:
    got = count_evenly_good_digit_dp(l, r)
    print(f"[{l}, {r}] -> {got} | mong đợi {expected}")
    assert got == expected

[18, 22] -> 3 | mong đợi 3
[98, 101] -> 2 | mong đợi 2
[1, 10] -> 5 | mong đợi 5


## Độ phức tạp của Digit DP

- Số trạng thái: `pos` có `n + 1` giá trị (`n <= 16`), `parity` 2, `tight` 2, `started` 2 → tối đa khoảng `17 * 8 = 136` trạng thái (thực tế còn ít hơn vì `tight = True` chỉ xảy ra trên một đường duy nhất).
- Mỗi trạng thái thử tối đa 10 chữ số.
- Thời gian: `O(D * 2 * 2 * 2 * 10)` = `O(D)` với hằng số khoảng 80. Bộ nhớ: `O(D)` cho cache và độ sâu đệ quy.

Cell dưới in số trạng thái thực sự được tính (`misses`) khi `x = 10^15` — chỉ vài chục!

In [16]:
big = 10 ** 15
print("F(10^15) =", count_upto_digit_dp(big))
print("Cache:", count_upto_digit_dp.last_cache_info)
print("Đáp án cho l = 1, r = 10^15:", count_evenly_good_digit_dp(1, big))

F(10^15) = 500000000000000
Cache: CacheInfo(hits=385, misses=63, maxsize=None, currsize=63)
Đáp án cho l = 1, r = 10^15: 500000000000000


# 8. Mô phỏng Digit DP với `x = 22`

`s = "22"`, `n = 2`. Lời gọi gốc là `dfs(0, 0, True, False)`.

| Lời gọi | Tình huống | Thử `d` | Kết quả |
|---|---|---|---:|
| `dfs(0, 0, True, False)` | chưa đặt gì, đang bám sát `"22"` | `0, 1, 2` | `5 + 5 + 2 = 12` |
| `dfs(1, 0, False, False)` | chọn `d=0` đầu tiên → số 0 đệm, ta đang dựng **số 1 chữ số** | `0..9` | `5` (là `1, 3, 5, 7, 9`) |
| `dfs(1, 0, False, True)` | chọn `d=1` → prefix `"1"`, parity 0, đã nhỏ hơn `x` nên tự do | `0..9` | `5` (là `11, 13, 15, 17, 19`) |
| `dfs(1, 1, True, True)` | chọn `d=2` → prefix `"2"`, parity 1, vẫn bám sát | `0..2` | `2` (là `20, 22`) |

Đọc bảng:
- Nhánh `d = 0` không được tính số `0` (base case cần `started`): `"00"` cho `0`.
- Nhánh `d = 2` chỉ được thử chữ số cuối tới `2` vì vẫn `tight`. Trong `20, 21, 22`, chỉ `20, 22` tốt.
- `F(22) = 12`. Tương tự `F(17) = 9`, nên đáp án Ví dụ 1 là `12 - 9 = 3`.

Hàm `trace_digit_dp` dưới đây in **mỗi trạng thái mới** khi "vào" và khi "ra" (kèm đóng góp của từng chữ số `d`). Với `x = 101` bạn sẽ thấy nhiều lần **lấy từ memo**.

In [17]:
def trace_digit_dp(x: int) -> int:
    s = str(x)
    n = len(s)
    memo = {}

    def dfs(pos, parity, tight, started, depth):
        if pos == n:
            return 1 if (started and parity == 0) else 0
        key = (pos, parity, tight, started)
        indent = "    " * depth
        if key in memo:
            print(f"{indent}dfs{key} đã có trong memo -> {memo[key]}")
            return memo[key]
        up = int(s[pos]) if tight else 9
        print(f"{indent}vào dfs(pos={pos}, parity={parity}, tight={tight}, started={started}), thử d = 0..{up}")
        parts = []
        total = 0
        for d in range(up + 1):
            new_started = started or d != 0
            new_parity = parity ^ 1 if (new_started and d % 2 == 0) else parity
            got = dfs(pos + 1, new_parity, tight and d == up, new_started, depth + 1)
            parts.append(f"{d}:{got}")
            total += got
        print(f"{indent}ra  dfs{key} = {total}   [d:đóng góp -> {' '.join(parts)}]")
        memo[key] = total
        return total

    answer = dfs(0, 0, True, False, 0)
    print(f"=> F({x}) = {answer}")
    return answer


trace_digit_dp(22)
print()
trace_digit_dp(101)

vào dfs(pos=0, parity=0, tight=True, started=False), thử d = 0..2
    vào dfs(pos=1, parity=0, tight=False, started=False), thử d = 0..9
    ra  dfs(1, 0, False, False) = 5   [d:đóng góp -> 0:0 1:1 2:0 3:1 4:0 5:1 6:0 7:1 8:0 9:1]
    vào dfs(pos=1, parity=0, tight=False, started=True), thử d = 0..9
    ra  dfs(1, 0, False, True) = 5   [d:đóng góp -> 0:0 1:1 2:0 3:1 4:0 5:1 6:0 7:1 8:0 9:1]
    vào dfs(pos=1, parity=1, tight=True, started=True), thử d = 0..2
    ra  dfs(1, 1, True, True) = 2   [d:đóng góp -> 0:1 1:0 2:1]
ra  dfs(0, 0, True, False) = 12   [d:đóng góp -> 0:5 1:5 2:2]
=> F(22) = 12

vào dfs(pos=0, parity=0, tight=True, started=False), thử d = 0..1
    vào dfs(pos=1, parity=0, tight=False, started=False), thử d = 0..9
        vào dfs(pos=2, parity=0, tight=False, started=False), thử d = 0..9
        ra  dfs(2, 0, False, False) = 5   [d:đóng góp -> 0:0 1:1 2:0 3:1 4:0 5:1 6:0 7:1 8:0 9:1]
        vào dfs(pos=2, parity=0, tight=False, started=True), thử d = 0..9
        ra  

51

### Điều đáng để ý trong trace

Trong trace của `x = 101`, mọi trạng thái **không bám sát** (`tight=False`) đều trả về đúng một nửa số cách điền: tầng `pos = 2` (còn 1 chữ số tự do) luôn ra `5` trên 10 cách; tầng `pos = 1` với `started=False` ra `50` trên 100 cách. Kết quả này **không phụ thuộc parity** đang có!

Đó không phải trùng hợp. Bước đột phá tiếp theo biến quan sát này thành một định luật, giúp ta **bỏ hẳn đệ quy**.

# 9. Bước đột phá 3 — Luật "một nửa"

## Thí nghiệm

Đếm số tốt trong **tất cả các số có đúng `k` chữ số**:
- `k = 1`: các số `1..9`.
- `k >= 2`: các số `10^(k-1) .. 10^k - 1`.

Dự đoán trước khi chạy: có phải luôn khoảng một nửa?

In [18]:
print(" k | số các số k chữ số | số tốt | tỉ lệ")
for k in range(1, 6):
    lo = 1 if k == 1 else 10 ** (k - 1)
    hi = 10 ** k - 1
    good = count_evenly_good_bruteforce(lo, hi)
    total = hi - lo + 1
    print(f"{k:>2} | {total:>18} | {good:>6} | {good / total:.4f}")

 k | số các số k chữ số | số tốt | tỉ lệ
 1 |                  9 |      5 | 0.5556
 2 |                 90 |     45 | 0.5000
 3 |                900 |    450 | 0.5000
 4 |               9000 |   4500 | 0.5000


 5 |              90000 |  45000 | 0.5000


## Giải thích bằng phép "ghép cặp"

Kết quả: với `k >= 2` luôn **đúng một nửa** (`45/90`, `450/900`, ...); riêng `k = 1` là `5/9`.

**Lý do (ghép cặp chữ số cuối):** ghép các chữ số thành 5 cặp `0↔1, 2↔3, 4↔5, 6↔7, 8↔9` (tức `d ↔ d ^ 1`). Trong mỗi cặp có đúng một chữ số chẵn và một chữ số lẻ. Lấy một số bất kỳ, **đổi chữ số cuối sang "bạn cặp" của nó**:
- prefix không đổi,
- chữ số cuối đổi từ chẵn sang lẻ hoặc ngược lại → tổng số chữ số chẵn thay đổi đúng 1 → **parity lật**.

Vậy phép đổi này ghép mỗi số tốt với đúng một số không tốt (và ngược lại) → hai nhóm **bằng nhau**.

Phép ghép cặp chỉ cần chữ số cuối là **tự do `0..9`**. Với `k = 1`, chữ số duy nhất cũng là chữ số đầu, không được là `0` → chỉ còn `1..9` (5 lẻ, 4 chẵn) → mất cân bằng, ra `5/9`.

## Phát biểu tổng quát

> Với một prefix cố định (parity bất kỳ), nếu còn **`m >= 1` chữ số tự do** (mỗi chữ số chạy `0..9`), thì trong `10^m` cách điền có **đúng `10^m / 2`** cách cho tổng parity bằng 0, và `10^m / 2` cách cho parity bằng 1.
> Nếu `m = 0` (hết chỗ điền) thì chỉ có 1 cách, và cách đó tốt ⇔ parity hiện tại bằng 0.

Cách chứng minh thứ hai bằng tổ hợp (N6): số cách có số chẵn chữ số chẵn là `sum_{j chẵn} C(m, j) * 5^m = 5^m * 2^(m-1) = 10^m / 2`, vì với `m >= 1` tổng các `C(m, j)` với `j` chẵn bằng `2^(m-1)`.

In [19]:
from math import comb


def ways_free_comb(m: int, need: int) -> int:
    """Số chuỗi m chữ số (0..9) có số chữ số chẵn ≡ need (mod 2) — tính bằng tổ hợp."""
    return sum(comb(m, j) * 5 ** m for j in range(m + 1) if j % 2 == need)


# Ghép cặp: prefix '2' (parity 1), đổi chữ số cuối d <-> d ^ 1
for d in range(0, 10, 2):
    a, b = 20 + d, 20 + (d ^ 1)
    print(f"{a} (tốt: {is_evenly_good(a)!s:<5}) <-> {b} (tốt: {is_evenly_good(b)})")

print()
# Dòng m = 0 cho thấy vì sao m = 0 phải xử lý riêng: không còn chữ số tự do thì không "chia đôi" được.
for m in range(0, 6):
    print(f"m = {m}: parity 0 -> {ways_free_comb(m, 0):>6}, parity 1 -> {ways_free_comb(m, 1):>6}, 10^m // 2 = {10 ** m // 2}")

20 (tốt: True ) <-> 21 (tốt: False)
22 (tốt: True ) <-> 23 (tốt: False)
24 (tốt: True ) <-> 25 (tốt: False)
26 (tốt: True ) <-> 27 (tốt: False)
28 (tốt: True ) <-> 29 (tốt: False)

m = 0: parity 0 ->      1, parity 1 ->      0, 10^m // 2 = 0
m = 1: parity 0 ->      5, parity 1 ->      5, 10^m // 2 = 5
m = 2: parity 0 ->     50, parity 1 ->     50, 10^m // 2 = 50
m = 3: parity 0 ->    500, parity 1 ->    500, 10^m // 2 = 500
m = 4: parity 0 ->   5000, parity 1 ->   5000, 10^m // 2 = 5000
m = 5: parity 0 ->  50000, parity 1 ->  50000, 10^m // 2 = 50000


# 10. Cách 3 — Toán đếm (không đệ quy)

Nhờ luật "một nửa", ta đếm `F(x)` (với `x >= 1`) bằng cách cộng các **nhóm** như N7, mỗi nhóm đếm bằng công thức.

Đặt `s = str(x)`, `D = len(s)` và hàm phụ:

```text
ways_free(m, need) = 10^m / 2                  nếu m >= 1
                   = 1 nếu need == 0, ngược lại 0   nếu m == 0
```

`ways_free(m, need)` = số cách điền `m` chữ số tự do để **phần đuôi** có parity `need`. Muốn cả số tốt, phần đuôi phải có parity **bằng** parity của prefix (vì `prefix + đuôi ≡ 0`).

## Phần A — các số có ít chữ số hơn `x`

Với mỗi độ dài `k = 1 .. D - 1`: chữ số đầu `c` chạy `1..9`, còn `k - 1` chữ số tự do:

```text
số tốt có k chữ số = tổng theo c = 1..9 của ways_free(k - 1, parity(c))
                   = 5                (k = 1)
                   = 9 * 10^(k-1) / 2 (k >= 2)
```

(`parity(c) = 1` nếu `c` chẵn, `0` nếu `c` lẻ.)

## Phần B — các số có đúng `D` chữ số và `<= x`

Đi dọc `s` từ trái sang phải, giữ `parity` của prefix **bám sát** `s[0..i-1]`. Tại vị trí `i`:
- Thử mọi chữ số `c < s[i]` (ở vị trí `i = 0` thì `c >= 1`). Đặt `c` thì số đã nhỏ hơn `x`, còn `m = D - 1 - i` chữ số tự do → cộng `ways_free(m, parity mới)`.
- Sau đó "đi tiếp" bằng chính `s[i]`: cập nhật `parity` theo `s[i]`.

Cuối cùng, chính `x` được cộng thêm 1 nếu parity của toàn bộ `x` bằng 0.

In [20]:
def ways_free(m: int, need: int) -> int:
    """Số cách điền m chữ số tự do sao cho phần đuôi có parity = need."""
    if m == 0:
        return 1 if need == 0 else 0
    return 10 ** m // 2


def even_bit(d: int) -> int:
    """1 nếu chữ số d chẵn, 0 nếu lẻ."""
    return 1 if d % 2 == 0 else 0


def count_upto_combinatorics(x: int) -> int:
    """F(x) bằng toán đếm: phần A (ít chữ số hơn) + phần B (đúng D chữ số, <= x)."""
    if x <= 0:
        return 0
    s = str(x)
    D = len(s)
    total = 0

    # Phần A: các số có k = 1 .. D-1 chữ số
    for k in range(1, D):
        for c in range(1, 10):
            total += ways_free(k - 1, even_bit(c))

    # Phần B: các số có đúng D chữ số và <= x
    parity = 0
    for i, ch in enumerate(s):
        cur = int(ch)
        lo = 1 if i == 0 else 0
        m = D - 1 - i
        for c in range(lo, cur):
            total += ways_free(m, parity ^ even_bit(c))
        parity ^= even_bit(cur)

    if parity == 0:          # chính x
        total += 1
    return total


def count_evenly_good_combinatorics(l: int, r: int) -> int:
    return count_upto_combinatorics(r) - count_upto_combinatorics(l - 1)


for l, r, expected in examples:
    got = count_evenly_good_combinatorics(l, r)
    print(f"[{l}, {r}] -> {got} | mong đợi {expected}")
    assert got == expected
print("F(10^15) =", count_upto_combinatorics(10 ** 15))

[18, 22] -> 3 | mong đợi 3
[98, 101] -> 2 | mong đợi 2
[1, 10] -> 5 | mong đợi 5
F(10^15) = 500000000000000


## Mô phỏng Cách 3 với `x = 22`

`s = "22"`, `D = 2`.

**Phần A** (số 1 chữ số): `c = 1..9`, `m = 0` → chỉ `c` lẻ được tính → `+5`. `total = 5`.

**Phần B**:

| i | `s[i]` | parity prefix trước | thử `c < s[i]` | `m` tự do | cộng thêm | total |
|---:|---:|---:|---|---:|---|---:|
| 0 | 2 | 0 | `c = 1` → parity mới `0` | 1 | `ways_free(1, 0) = 5` (`11..19` lẻ) | 10 |
| 1 | 2 | 1 (vì `"2"`) | `c = 0` → parity mới `0`; `c = 1` → parity mới `1` | 0 | `1` (số `20`) + `0` (số `21`) | 11 |
| cuối | — | 0 (vì `"22"`) | chính `x = 22` | — | `+1` | 12 |

→ `F(22) = 12`, khớp với Digit DP.

Hàm `trace_combinatorics` in từng bước. Thử thêm `x = 253` để thấy nhóm nào đóng góp bao nhiêu (so với bảng N7).

In [21]:
def trace_combinatorics(x: int) -> int:
    s = str(x)
    D = len(s)
    total = 0
    print(f"x = {x}, D = {D}")
    for k in range(1, D):
        part = sum(ways_free(k - 1, even_bit(c)) for c in range(1, 10))
        total += part
        print(f"  Phần A: số có {k} chữ số -> +{part}, total = {total}")
    parity = 0
    for i, ch in enumerate(s):
        cur = int(ch)
        lo = 1 if i == 0 else 0
        m = D - 1 - i
        for c in range(lo, cur):
            need = parity ^ even_bit(c)
            add = ways_free(m, need)
            total += add
            print(f"  Phần B: i={i}, prefix '{s[:i]}' + c={c} (parity {need}), m={m} tự do -> +{add}, total = {total}")
        parity ^= even_bit(cur)
        print(f"          bám sát s[{i}]={cur}: parity prefix '{s[:i + 1]}' = {parity}")
    if parity == 0:
        total += 1
        print(f"  Chính x = {x} có parity 0 -> +1, total = {total}")
    else:
        print(f"  Chính x = {x} có parity 1 -> không cộng")
    print(f"=> F({x}) = {total}")
    return total


trace_combinatorics(22)
print()
trace_combinatorics(253)

x = 22, D = 2
  Phần A: số có 1 chữ số -> +5, total = 5
  Phần B: i=0, prefix '' + c=1 (parity 0), m=1 tự do -> +5, total = 10
          bám sát s[0]=2: parity prefix '2' = 1
  Phần B: i=1, prefix '2' + c=0 (parity 0), m=0 tự do -> +1, total = 11
  Phần B: i=1, prefix '2' + c=1 (parity 1), m=0 tự do -> +0, total = 11
          bám sát s[1]=2: parity prefix '22' = 0
  Chính x = 22 có parity 0 -> +1, total = 12
=> F(22) = 12

x = 253, D = 3
  Phần A: số có 1 chữ số -> +5, total = 5
  Phần A: số có 2 chữ số -> +45, total = 50
  Phần B: i=0, prefix '' + c=1 (parity 0), m=2 tự do -> +50, total = 100
          bám sát s[0]=2: parity prefix '2' = 1
  Phần B: i=1, prefix '2' + c=0 (parity 0), m=1 tự do -> +5, total = 105
  Phần B: i=1, prefix '2' + c=1 (parity 1), m=1 tự do -> +5, total = 110
  Phần B: i=1, prefix '2' + c=2 (parity 0), m=1 tự do -> +5, total = 115
  Phần B: i=1, prefix '2' + c=3 (parity 1), m=1 tự do -> +5, total = 120
  Phần B: i=1, prefix '2' + c=4 (parity 0), m=1 tự do -> +

127

## Độ phức tạp của Cách 3

- Phần A: `D - 1` độ dài × 9 chữ số đầu → `O(9D)`.
- Phần B: `D` vị trí × tối đa 10 chữ số → `O(10D)`.
- Tổng: **`O(D)`** thời gian, `O(D)` bộ nhớ (chuỗi `s`). Không đệ quy, không cache.

Thực ra phần A còn có dạng đóng: `5 + 45 + 450 + ... = 10^(D-1) / 2` (với `D >= 2`). Ví dụ `D = 3`: `5 + 45 = 50 = 100 / 2`. Bạn có thể kiểm tra tay!

# 11. Bước đột phá 4 — Mỗi khối 10 số có đúng 5 số tốt

Nhớ lại thí nghiệm ở mục 2: `[1..9]`, `[10..19]`, `[20..29]`, `[100..109]`, `[250..259]` đều có 5 số tốt. Giờ ta chứng minh được ngay bằng luật "một nửa" với `m = 1`:

Xét khối `[10q, 10q + 9]` với `q >= 1`. Mọi số trong khối có dạng `10q + d` với `d = 0..9`: **prefix là các chữ số của `q`**, chữ số cuối `d` tự do.
- Gọi `p` = parity số chữ số chẵn của `q`.
- `10q + d` tốt ⇔ `p ^ even_bit(d) == 0`.
- Có 5 chữ số chẵn và 5 chữ số lẻ → **đúng 5** giá trị `d` thỏa.

Với `q = 0` (khối `[0..9]`): các số chính là `d` (không có prefix, coi `p = 0`) → tốt ⇔ `d` lẻ → `1, 3, 5, 7, 9` = 5 số; số `0` không được tính (và đề cũng không xét `0`). Vẫn đúng 5!

Cell dưới kiểm tra 30 khối đầu tiên và vài khối ở rất xa.

In [22]:
counts = []
for q in range(30):
    block_good = sum(1 for x in range(10 * q, 10 * q + 10) if x >= 1 and is_evenly_good(x))
    counts.append(block_good)
print("Số tốt trong 30 khối đầu:", counts)

for q in [12345, 99999999, 10 ** 13 + 7]:
    block_good = sum(1 for x in range(10 * q, 10 * q + 10) if is_evenly_good(x))
    print(f"khối q = {q}: {block_good} số tốt")

Số tốt trong 30 khối đầu: [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]
khối q = 12345: 5 số tốt
khối q = 99999999: 5 số tốt
khối q = 10000000000007: 5 số tốt


## Cách 4 — Công thức khối 10

Viết `x = 10q + t` (`q, t = divmod(x, 10)`, `0 <= t <= 9`). Đoạn `[0, x]` gồm:
- `q` khối đầy đủ `[0..9], [10..19], ..., [10(q-1) .. 10(q-1)+9]` → mỗi khối 5 số tốt → **`5q`**.
- Một khối dở dang `[10q .. 10q + t]` → chỉ cần đếm `d = 0..t` sao cho `p ^ even_bit(d) == 0`, với `p` = parity số chữ số chẵn của `q`:
  - nếu `p = 0`: cần `d` **lẻ** trong `[0, t]` → có `(t + 1) // 2` số;
  - nếu `p = 1`: cần `d` **chẵn** trong `[0, t]` → có `t // 2 + 1` số.

```text
F(x) = 5 * q + tail(p, t)
```

Lưu ý `q = 0` phải cho `p = 0` (không có chữ số nào), **không phải** coi `q` là chữ số `"0"`. Vòng `while q > 0` tự nhiên làm đúng điều này. Nhờ vậy `F(0) = 0` mà không cần xử lý riêng.

Kiểm tra tay Ví dụ 1:
- `F(22)`: `q = 2, t = 2`, `"2"` có 1 chữ số chẵn → `p = 1` → `t // 2 + 1 = 2` (là `20, 22`) → `F = 10 + 2 = 12`.
- `F(17)`: `q = 1, t = 7`, `p = 0` → `(7 + 1) // 2 = 4` (là `11, 13, 15, 17`) → `F = 5 + 4 = 9`.
- Đáp án `12 - 9 = 3`. ✔

In [23]:
def even_parity_of(q: int) -> int:
    """Parity số chữ số chẵn của q. Quy ước q = 0 -> 0 (không có chữ số nào)."""
    p = 0
    while q > 0:
        q, d = divmod(q, 10)
        if d % 2 == 0:
            p ^= 1
    return p


def count_upto_blocks(x: int) -> int:
    """F(x) = 5 * (x // 10) + phần đuôi của khối cuối."""
    q, t = divmod(x, 10)
    p = even_parity_of(q)
    tail = (t + 1) // 2 if p == 0 else t // 2 + 1
    return 5 * q + tail


def count_evenly_good_blocks(l: int, r: int) -> int:
    return count_upto_blocks(r) - count_upto_blocks(l - 1)


for l, r, expected in examples:
    got = count_evenly_good_blocks(l, r)
    print(f"[{l}, {r}] -> {got} | mong đợi {expected}")
    assert got == expected
print("F(0) =", count_upto_blocks(0), "| F(10^15) =", count_upto_blocks(10 ** 15))

[18, 22] -> 3 | mong đợi 3
[98, 101] -> 2 | mong đợi 2
[1, 10] -> 5 | mong đợi 5
F(0) = 0 | F(10^15) = 500000000000000


In [24]:
def trace_blocks(l: int, r: int) -> int:
    values = {}
    for x in (r, l - 1):
        q, t = divmod(x, 10)
        p = even_parity_of(q)
        if p == 0:
            tail = (t + 1) // 2
            rule = "p = 0 -> cần chữ số cuối LẺ"
        else:
            tail = t // 2 + 1
            rule = "p = 1 -> cần chữ số cuối CHẴN"
        values[x] = 5 * q + tail
        print(f"  F({x}): q = {q}, t = {t}, {rule} trong [0, {t}] -> tail = {tail}; F = 5*{q} + {tail} = {values[x]}")
    answer = values[r] - values[l - 1]
    print(f"  => đáp án [{l}, {r}] = F({r}) - F({l - 1}) = {answer}")
    return answer


for l, r, expected in examples:
    print(f"Ví dụ l = {l}, r = {r} (mong đợi {expected}):")
    assert trace_blocks(l, r) == expected

Ví dụ l = 18, r = 22 (mong đợi 3):
  F(22): q = 2, t = 2, p = 1 -> cần chữ số cuối CHẴN trong [0, 2] -> tail = 2; F = 5*2 + 2 = 12
  F(17): q = 1, t = 7, p = 0 -> cần chữ số cuối LẺ trong [0, 7] -> tail = 4; F = 5*1 + 4 = 9
  => đáp án [18, 22] = F(22) - F(17) = 3
Ví dụ l = 98, r = 101 (mong đợi 2):
  F(101): q = 10, t = 1, p = 1 -> cần chữ số cuối CHẴN trong [0, 1] -> tail = 1; F = 5*10 + 1 = 51
  F(97): q = 9, t = 7, p = 0 -> cần chữ số cuối LẺ trong [0, 7] -> tail = 4; F = 5*9 + 4 = 49
  => đáp án [98, 101] = F(101) - F(97) = 2
Ví dụ l = 1, r = 10 (mong đợi 5):
  F(10): q = 1, t = 0, p = 0 -> cần chữ số cuối LẺ trong [0, 0] -> tail = 0; F = 5*1 + 0 = 5
  F(0): q = 0, t = 0, p = 0 -> cần chữ số cuối LẺ trong [0, 0] -> tail = 0; F = 5*0 + 0 = 0
  => đáp án [1, 10] = F(10) - F(0) = 5


## Độ phức tạp của Cách 4

- `divmod(x, 10)`: `O(1)` (với số nguyên cỡ `10^15`).
- `even_parity_of(q)`: một vòng qua các chữ số của `q` → `O(D)`.
- Gọi 2 lần (cho `r` và `l - 1`).

→ **Thời gian `O(D)`, bộ nhớ `O(1)`** — chỉ vài biến số nguyên. Đây là lời giải ngắn và nhanh nhất; Digit DP (Cách 2) là lời giải **tổng quát** nhất (dùng được cho rất nhiều bài khác).

# 12. Phiên bản nộp LeetCode

Ta nộp Cách 4 vì ngắn, `O(D)` và không có đệ quy. Hàm phụ `count_upto` được viết bên trong method để code tự đủ (không phụ thuộc các hàm minh họa phía trên).

Lưu ý: tên method và tham số phải khớp đúng skeleton: `countEvenlyGoodIntegers(self, l: int, r: int) -> int`.

In [25]:
class Solution:
    def countEvenlyGoodIntegers(self, l: int, r: int) -> int:
        def count_upto(x: int) -> int:
            # Số lượng số evenly good trong [1, x]
            q, t = divmod(x, 10)
            parity = 0
            y = q
            while y > 0:                 # parity số chữ số chẵn của q (q = 0 -> 0)
                y, d = divmod(y, 10)
                if d % 2 == 0:
                    parity ^= 1
            if parity == 0:
                tail = (t + 1) // 2      # chữ số cuối phải lẻ
            else:
                tail = t // 2 + 1        # chữ số cuối phải chẵn
            return 5 * q + tail

        return count_upto(r) - count_upto(l - 1)

In [26]:
sol = Solution()

# Các ví dụ trong đề
assert sol.countEvenlyGoodIntegers(18, 22) == 3
assert sol.countEvenlyGoodIntegers(98, 101) == 2
assert sol.countEvenlyGoodIntegers(1, 10) == 5

# Edge cases nhỏ
assert sol.countEvenlyGoodIntegers(1, 1) == 1      # 1: 0 chữ số chẵn
assert sol.countEvenlyGoodIntegers(2, 2) == 0      # 2: 1 chữ số chẵn
assert sol.countEvenlyGoodIntegers(10, 10) == 0    # 10: chữ số 0 là chẵn
assert sol.countEvenlyGoodIntegers(11, 11) == 1
assert sol.countEvenlyGoodIntegers(1, 9) == 5
assert sol.countEvenlyGoodIntegers(100, 100) == 1  # hai chữ số 0

# Biên lớn
big = 10 ** 15
assert sol.countEvenlyGoodIntegers(big, big) == 0          # 1 và 15 chữ số 0 -> 15 chữ số chẵn
assert sol.countEvenlyGoodIntegers(1, big) == 5 * 10 ** 14
for lo in [big - 50, 999_999_999_999_000, 123_456_789_012_345]:
    hi = min(lo + 50, big)
    assert sol.countEvenlyGoodIntegers(lo, hi) == count_evenly_good_bruteforce(lo, hi)

print("Tất cả test của Solution đều PASS!")
print("Kết quả cho [1, 10^15]:", sol.countEvenlyGoodIntegers(1, big))

Tất cả test của Solution đều PASS!
Kết quả cho [1, 10^15]: 500000000000000


# 13. Kiểm thử đối chiếu (cross-check)

Ba lời giải nhanh dựa trên ba lập luận khác nhau. Nếu cả ba **cùng khớp với brute force** trên rất nhiều input ngẫu nhiên, ta tin tưởng hơn nhiều so với chỉ chạy 3 ví dụ của đề.

- Phần 1: đoạn nhỏ ngẫu nhiên (seed cố định) → so với brute force.
- Phần 2: đoạn lớn tới `10^15` → brute force không chạy nổi, nên so **các cách nhanh với nhau**.
- Phần 3: cửa sổ ngắn ở vùng số lớn → vẫn so được với brute force.

(Trong lúc soạn notebook, bộ test ngoài notebook đã chạy hàng nghìn trường hợp; ở đây là phiên bản nhỏ, chạy nhanh.)

In [27]:
import random

random.seed(4060)
sol = Solution()
fast_methods = [
    ("digit_dp", count_evenly_good_digit_dp),
    ("combinatorics", count_evenly_good_combinatorics),
    ("blocks", count_evenly_good_blocks),
    ("Solution", sol.countEvenlyGoodIntegers),
]

# Phần 1: đoạn nhỏ, so với brute force
n_small = 400
for _ in range(n_small):
    l = random.randint(1, 3000)
    r = random.randint(l, min(3000, l + 400))
    expected = count_evenly_good_bruteforce(l, r)
    for name, f in fast_methods:
        assert f(l, r) == expected, (name, l, r)

# Phần 2: đoạn lớn, so các cách nhanh với nhau
n_big = 300
for _ in range(n_big):
    l = random.randint(1, 10 ** 15)
    r = random.randint(l, 10 ** 15)
    results = {f(l, r) for _, f in fast_methods}
    assert len(results) == 1, (l, r, results)

# Phần 3: cửa sổ ngắn ở vùng số lớn, so với brute force
n_window = 100
for _ in range(n_window):
    l = random.randint(1, 10 ** 15 - 100)
    r = l + random.randint(0, 100)
    expected = count_evenly_good_bruteforce(l, r)
    for name, f in fast_methods:
        assert f(l, r) == expected, (name, l, r)

print(f"PASS: {n_small} đoạn nhỏ + {n_big} đoạn lớn + {n_window} cửa sổ lớn, cả 4 lời giải nhanh đều khớp.")

PASS: 400 đoạn nhỏ + 300 đoạn lớn + 100 cửa sổ lớn, cả 4 lời giải nhanh đều khớp.


# 14. Vì sao các lời giải đúng?

## Nền chung: `F(r) - F(l - 1)`

`F(x)` đếm `[1, x]`. Với `l >= 1`, các số trong `[1, r]` chia làm hai phần rời nhau `[1, l-1]` và `[l, r]`, nên `F(r) = F(l - 1) + đáp án`.

## Cách 2 — Digit DP

**Bất biến:** `dfs(pos, parity, tight, started)` bằng số cách điền các vị trí `pos..n-1` sao cho số hoàn chỉnh `<= x`, khác 0, và có tổng số chữ số chẵn (thật) chẵn — biết rằng phần đã điền có parity `parity`, có bám sát `x` hay không (`tight`), và đã bắt đầu hay chưa (`started`).

- **Không bỏ sót, không đếm trùng:** mỗi số `y` trong `[0, x]` ứng với **đúng một** chuỗi `n` ký tự (đệm số 0 ở đầu), tức đúng một đường đi trong cây lựa chọn. Giới hạn `up` bảo đảm không bao giờ tạo ra chuỗi lớn hơn `s`.
- **Đúng tính chất:** số 0 đệm không lật parity nhờ `started`; base case loại `y = 0` và chỉ nhận parity 0.
- **Memo hợp lệ:** kết quả chỉ phụ thuộc 4 tham số (Bước đột phá 2: phần đã điền chỉ ảnh hưởng qua parity).

## Cách 3 — Toán đếm

Mọi số `y` trong `[1, x]` hoặc có ít chữ số hơn `x` (phần A), hoặc có đúng `D` chữ số; khi đó hoặc `y = x`, hoặc tồn tại **vị trí đầu tiên `i`** mà `y` có chữ số nhỏ hơn `s[i]` (phần B, N7). Các nhóm này rời nhau và phủ hết. Trong mỗi nhóm, phần đuôi tự do được đếm chính xác bằng luật "một nửa" (đã chứng minh bằng ghép cặp `d ↔ d ^ 1` và bằng tổ hợp).

## Cách 4 — Khối 10

`[0, x]` = `q` khối đầy đủ + một khối dở dang `[10q, 10q + t]`. Mỗi khối đầy đủ có đúng 5 số tốt (5 chữ số cuối chẵn, 5 lẻ; số `0` ở khối đầu không tốt theo quy ước `p(0) = 0`). Khối dở dang được đếm trực tiếp bằng công thức số lẻ/số chẵn trong `[0, t]`.

# 15. Độ phức tạp và so sánh các cách

Ký hiệu: `N = r - l + 1` (số lượng số trong đoạn), `D` = số chữ số của `r` (`D <= 16`).

| Cách | Ý tưởng | Thời gian | Bộ nhớ | Ghi chú |
|---|---|---|---|---|
| 1. Brute force | Duyệt từng số, đếm chữ số chẵn | `O(N * D)` | `O(D)` | Chỉ dùng làm trọng tài / kiểm thử |
| 2. Digit DP | `dfs(pos, parity, tight, started)` + `@cache` | `O(D * 8 * 10)` | `O(D)` | Tổng quát nhất, áp dụng cho nhiều bài |
| 3. Toán đếm | Phần A + phần B + luật "một nửa" | `O(D * 10)` | `O(D)` | Không đệ quy; cần hiểu luật một nửa |
| 4. Khối 10 | `F(x) = 5 * (x // 10) + tail` | `O(D)` | `O(1)` | Ngắn nhất, nhanh nhất cho riêng bài này |

Cả ba cách nhanh chỉ tốn vài trăm phép tính cho `r = 10^15`, nên **Python thuần chạy thoải mái** trên LeetCode. Nếu dùng Cách 2 trong Python, đệ quy sâu tối đa 17 tầng — không lo giới hạn recursion.

**Nên dùng cách nào?**
- Trong phỏng vấn: trình bày Cách 2 trước (cho thấy bạn biết khung digit DP), sau đó nêu quan sát Cách 4 như một tối ưu đẹp.
- Khi chỉ cần AC nhanh: Cách 4.

# 16. Những lỗi thường gặp

## Lỗi 1 — Quên rằng `0` là chữ số chẵn

Sai:

```python
even = sum(1 for ch in str(x) if ch in "2468")     # thiếu "0"
```

Với `101`, code sai đếm 0 chữ số chẵn → "tốt", nhưng thật ra `101` có 1 chữ số chẵn (`0`) → không tốt. Ví dụ 2 sẽ ra 3 thay vì 2.

Đúng: `ch in "02468"` hoặc `int(ch) % 2 == 0`.

## Lỗi 2 — Trừ `F(l)` thay vì `F(l - 1)`

Sai: `F(r) - F(l)` làm mất chính số `l` khi `l` là số tốt. Ví dụ 3: `F(10) - F(1) = 5 - 1 = 4` thay vì 5.

## Lỗi 3 — Nhầm "chữ số chẵn" với "số chẵn"

Đề không hỏi `x % 2 == 0`. Nhưng hãy xem điều bất ngờ: đếm **số chẵn** thay vì số tốt thì
- `[18, 22]`: số chẵn là `18, 20, 22` → 3 — **trùng** đáp án;
- `[98, 101]`: số chẵn là `98, 100` → 2 — **trùng**;
- `[1, 10]`: số chẵn là `2, 4, 6, 8, 10` → 5 — lại **trùng**!

Cả ba ví dụ của đề đều **không bắt được** lỗi này. Lý do sâu xa: mỗi khối 10 số có đúng 5 số chẵn **và** đúng 5 số tốt (Cách 4), nên trên các khối đầy đủ hai cách đếm luôn bằng nhau (ví dụ `[1, 30]` cả hai đều ra 15). Chúng chỉ lệch nhau ở khối dở dang, ví dụ `[11, 19]`: 4 số chẵn nhưng 5 số tốt.

Bài học lớn: **đừng chỉ tin vào ví dụ của đề** — hãy so với brute force trên input ngẫu nhiên.

In [28]:
def wrong_count_missing_zero(l, r):
    return sum(1 for x in range(l, r + 1) if sum(1 for ch in str(x) if ch in "2468") % 2 == 0)

def wrong_count_even_numbers(l, r):
    return sum(1 for x in range(l, r + 1) if x % 2 == 0)

print("Lỗi 1 (thiếu '0'):     [98, 101] ->", wrong_count_missing_zero(98, 101), "| đúng:", count_evenly_good_bruteforce(98, 101))
print("Lỗi 2 (F(r) - F(l)):   [1, 10]   ->", count_upto_blocks(10) - count_upto_blocks(1), "| đúng:", count_evenly_good_blocks(1, 10))
for l, r, expected in examples:
    print(f"Lỗi 3 (đếm số chẵn):   [{l}, {r}] -> {wrong_count_even_numbers(l, r)} | đúng: {expected}")
for l, r in [(1, 30), (11, 19), (1, 1)]:
    print(f"Lỗi 3 (đếm số chẵn):   [{l}, {r}] -> {wrong_count_even_numbers(l, r)} | đúng: {count_evenly_good_bruteforce(l, r)}")

Lỗi 1 (thiếu '0'):     [98, 101] -> 3 | đúng: 2
Lỗi 2 (F(r) - F(l)):   [1, 10]   -> 4 | đúng: 5
Lỗi 3 (đếm số chẵn):   [18, 22] -> 3 | đúng: 3
Lỗi 3 (đếm số chẵn):   [98, 101] -> 2 | đúng: 2
Lỗi 3 (đếm số chẵn):   [1, 10] -> 5 | đúng: 5
Lỗi 3 (đếm số chẵn):   [1, 30] -> 15 | đúng: 15
Lỗi 3 (đếm số chẵn):   [11, 19] -> 4 | đúng: 5
Lỗi 3 (đếm số chẵn):   [1, 1] -> 0 | đúng: 1


## Lỗi 4 — Bỏ cờ `started` trong digit DP (leading zero)

Nhiều lời giải digit DP viết `dfs(pos, parity, tight)` và coi số 0 đệm là chữ số chẵn. Với bài này, kết quả **vẫn đúng** — vì sao?

- Gọi `x` có `D >= 2` chữ số. Các số ngắn hơn `x` chính là `[0, 10^(D-1) - 1]`, được viết thành chuỗi `D` ký tự bắt đầu bằng `0`.
- Đếm theo kiểu "đệm số 0": chữ số đầu là `0` (chẵn), còn `D - 1` chữ số tự do → theo luật một nửa có đúng `10^(D-1) / 2` chuỗi tốt.
- Đếm thật: `5 + 45 + 450 + ... = 10^(D-1) / 2`. **Hai con số bằng nhau**, dù tập số được đếm khác nhau (kiểu đệm coi `"05"` là không tốt nhưng lại coi `"00"`, `"02"` là tốt).

Đó là **may mắn của bài này**, không phải kỹ thuật đúng. Đổi đề một chút — "đếm số có **đúng 0** chữ số chẵn trong `[1, 100]`" — thì bản thiếu `started` ra `0`, còn đáp án đúng là `30` (`5` số một chữ số + `25` số hai chữ số toàn chữ số lẻ). Cell dưới chứng minh cả hai điều.

In [29]:
from functools import cache


def count_upto_no_started(x: int) -> int:
    """Digit DP KHÔNG có started: coi số 0 đệm là chữ số chẵn, đếm trên [0, x]."""
    s = str(x)
    n = len(s)

    @cache
    def dfs(pos, parity, tight):
        if pos == n:
            return 1 if parity == 0 else 0
        up = int(s[pos]) if tight else 9
        return sum(dfs(pos + 1, parity ^ (1 if d % 2 == 0 else 0), tight and d == up) for d in range(up + 1))

    return dfs(0, 0, True)


# 1) Với bài gốc: tình cờ vẫn khớp
same = all(count_upto_no_started(x) == count_upto_blocks(x) for x in range(0, 3000))
print("Bài gốc, x = 0..2999: bản thiếu started khớp đáp án đúng?", same)


# 2) Biến thể "đúng k chữ số chẵn": thiếu started thì sai
def count_upto_exactly_k_no_started(x: int, k: int) -> int:
    s = str(x)
    n = len(s)

    @cache
    def dfs(pos, cnt, tight):
        if cnt > k:
            return 0
        if pos == n:
            return 1 if cnt == k else 0
        up = int(s[pos]) if tight else 9
        return sum(dfs(pos + 1, cnt + (1 if d % 2 == 0 else 0), tight and d == up) for d in range(up + 1))

    return dfs(0, 0, True)


truth = sum(1 for x in range(1, 101) if even_digit_count(x) == 0)
print("Đúng 0 chữ số chẵn trong [1, 100]: đáp án đúng =", truth, "| bản thiếu started =", count_upto_exactly_k_no_started(100, 0))

Bài gốc, x = 0..2999: bản thiếu started khớp đáp án đúng? True
Đúng 0 chữ số chẵn trong [1, 100]: đáp án đúng = 30 | bản thiếu started = 0


## Lỗi 5 — Dùng chung cache giữa `F(r)` và `F(l - 1)`

Một kiểu viết phổ biến: `s` là biến bên ngoài, `dfs(pos, parity, tight)` có `@cache`, rồi gán `s = str(l - 1)` gọi một lần, gán `s = str(r)` gọi lần nữa. Nhưng cache chỉ nhớ theo **tham số** `(pos, parity, tight)`, không biết `s` đã đổi! Lần gọi thứ hai trả lại kết quả cũ → đáp án ra `0`.

Sửa: gọi `dfs.cache_clear()` giữa hai lần, hoặc (gọn hơn) định nghĩa `dfs` **bên trong** hàm `count_upto(x)` như Cách 2 để mỗi lần có cache mới.

In [30]:
from functools import cache


def buggy_shared_cache(l: int, r: int) -> int:
    s = ""

    @cache
    def dfs(pos, parity, tight):
        if pos == len(s):
            return 1 if parity == 0 else 0
        up = int(s[pos]) if tight else 9
        return sum(dfs(pos + 1, parity ^ (1 if d % 2 == 0 else 0), tight and d == up) for d in range(up + 1))

    s = str(l - 1)
    a = dfs(0, 0, True)
    # QUÊN: dfs.cache_clear()
    s = str(r)
    b = dfs(0, 0, True)
    return b - a


print("Bản dùng chung cache: [18, 22] ->", buggy_shared_cache(18, 22), "| [98, 101] ->", buggy_shared_cache(98, 101))
print("Bản đúng (cache riêng):  [18, 22] ->", count_evenly_good_digit_dp(18, 22), "| [98, 101] ->", count_evenly_good_digit_dp(98, 101))

Bản dùng chung cache: [18, 22] -> 0 | [98, 101] -> 0
Bản đúng (cache riêng):  [18, 22] -> 3 | [98, 101] -> 2


## Lỗi 6 — Ở Cách 4, coi `q = 0` là chữ số `"0"`

Nếu tính parity của `q` bằng `str(q)`, thì `q = 0` cho chuỗi `"0"` → 1 chữ số chẵn → `p = 1` → khối đầu tiên chọn các `d` **chẵn** `0, 2, 4, 6, 8` thay vì lẻ. Vừa đếm nhầm số `0`, vừa bỏ sót `1, 3, 5, 7, 9`. Ví dụ 3 ra `4` thay vì `5`.

Sửa: khi `q = 0` thì số **không có** phần đầu → `p = 0` (vòng `while q > 0` làm đúng điều này).

## Lỗi 7 — Dùng `/` thay vì `//`

`x / 10` trả về số thực (float). Với số lớn, float không biểu diễn chính xác mọi số nguyên, nên `int(x / 10)` có thể sai. Trong phạm vi `10^15` của đề thì may mắn chưa lộ, nhưng đây là thói quen nguy hiểm — luôn dùng `//` hoặc `divmod` cho số nguyên.

## Lỗi 8 — Cập nhật `tight` sai

`new_tight = tight and d == up` — phải so với **`up`** (chữ số của `x` khi đang bám sát). Viết `tight and d == 9` hoặc quên `tight and` sẽ cho phép tạo ra số lớn hơn `x` hoặc cắt mất các số hợp lệ.

In [31]:
def count_upto_blocks_wrong_q0(x: int) -> int:
    q, t = divmod(x, 10)
    p = sum(1 for ch in str(q) if ch in "02468") % 2     # SAI khi q = 0: str(0) = "0"
    tail = (t + 1) // 2 if p == 0 else t // 2 + 1
    return 5 * q + tail

print("Lỗi 6: [1, 10] ->", count_upto_blocks_wrong_q0(10) - count_upto_blocks_wrong_q0(0), "| đúng:", count_evenly_good_blocks(1, 10))

x = 10 ** 17 + 13                    # lớn hơn ràng buộc, chỉ để thấy nguy cơ của float
print("Lỗi 7: x // 10      =", x // 10)
print("       int(x / 10)  =", int(x / 10), "<- lệch vì float")

Lỗi 6: [1, 10] -> 4 | đúng: 5
Lỗi 7: x // 10      = 10000000000000001
       int(x / 10)  = 10000000000000002 <- lệch vì float


## Edge cases nên tự kiểm tra

| Trường hợp | Ví dụ | Kết quả | Vì sao dễ sai |
|---|---|---:|---|
| `l = r` là số tốt | `(1, 1)` | 1 | quên `l - 1` |
| `l = r` không tốt | `(2, 2)` | 0 | |
| Số có chữ số 0 | `(10, 10)` | 0 | quên 0 là chữ số chẵn |
| Nhiều chữ số 0 | `(100, 100)` | 1 | |
| `l = 1` | `(1, 10)` | 5 | `F(0)` phải bằng 0 |
| Biên trên | `(10^15, 10^15)` | 0 | 15 chữ số 0 → lẻ |
| Toàn bộ miền | `(1, 10^15)` | `5 * 10^14` | tràn số trong C++/Java |

Tất cả các trường hợp này đã có trong cell test của `Solution` ở mục 12.

# 17. Mẹo nhận dạng dạng bài (dấu hiệu nhận dạng)

## Dấu hiệu 1 — "Đếm số nguyên trong `[l, r]` có tính chất về **chữ số**", với `r` rất lớn (`10^9`, `10^15`, `10^18`)

→ Gần như chắc chắn là **Digit DP**. Bước đầu tiên luôn là `F(r) - F(l - 1)`.

## Dấu hiệu 2 — Tính chất chỉ phụ thuộc vào một "tóm tắt" nhỏ của prefix

Parity, tổng chữ số mod k, chữ số cuối cùng, tập chữ số đã dùng (bitmask), hiệu số chẵn − số lẻ... → đó chính là **trạng thái** của DP. Hỏi: "để quyết định phần còn lại, mình cần nhớ gì về phần đã viết?".

## Dấu hiệu 3 — Tính chất liên quan tới chữ số `0`, độ dài số, hay "chữ số đầu"

→ Bắt buộc có cờ **`started`** (leading zero). Hỏi: "số 0 đệm có làm thay đổi tính chất không?".

## Dấu hiệu 4 — Trạng thái chỉ là parity và các chữ số chia đều 5/5

→ Thử tìm **công thức đóng** (luật một nửa, khối 10). Hãy luôn in bảng số liệu nhỏ (như thí nghiệm "mỗi chục 5 số") — quy luật thường lộ ra rất nhanh.

## Khung Digit DP nên thuộc lòng

```python
def count_upto(x):
    s = str(x)
    @cache
    def dfs(pos, state, tight, started):
        if pos == len(s):
            return 1 if (started and accept(state)) else 0
        up = int(s[pos]) if tight else 9
        total = 0
        for d in range(up + 1):
            ns = started or d != 0
            new_state = step(state, d) if ns else state
            total += dfs(pos + 1, new_state, tight and d == up, ns)
        return total
    return dfs(0, init_state, True, False)

answer = count_upto(r) - count_upto(l - 1)
```

## Bài liên quan để luyện

| Bài | Tính chất cần đếm | Trạng thái gợi ý |
|---|---|---|
| 233. Number of Digit One | tổng số chữ số 1 | số chữ số 1 đã đặt |
| 357. Count Numbers with Unique Digits | các chữ số khác nhau | bitmask chữ số đã dùng |
| 600. Non-negative Integers without Consecutive Ones | dạng nhị phân không có "11" | bit trước đó |
| 902. Numbers At Most N Given Digit Set | chỉ dùng các chữ số cho trước | tight + started |
| 1012. Numbers With Repeated Digits | có chữ số lặp | bitmask |
| 2376. Count Special Integers | các chữ số khác nhau | bitmask |
| 2719. Count of Integers | tổng chữ số trong `[min_sum, max_sum]` | tổng chữ số |
| 2801. Count Stepping Numbers in Range | hai chữ số kề nhau lệch 1 | chữ số trước |
| 2827. Number of Beautiful Integers in the Range | số chữ số chẵn = số chữ số lẻ **và** chia hết cho `k` | hiệu chẵn−lẻ, số dư mod `k` |

Bài **2827** là "anh em ruột" của bài này: cũng đếm chữ số chẵn/lẻ, nhưng thêm điều kiện chia hết. Cell dưới là một bản mini, kiểm tra với brute force.

In [32]:
from functools import cache
import random


def count_beautiful_upto(x: int, k: int) -> int:
    """Số lượng y trong [1, x]: (số chữ số chẵn == số chữ số lẻ) và y % k == 0."""
    if x <= 0:
        return 0
    s = str(x)
    n = len(s)

    @cache
    def dfs(pos, diff, rem, tight, started):
        if pos == n:
            return 1 if (started and diff == 0 and rem == 0) else 0
        up = int(s[pos]) if tight else 9
        total = 0
        for d in range(up + 1):
            ns = started or d != 0
            nd = diff + ((1 if d % 2 == 0 else -1) if ns else 0)
            total += dfs(pos + 1, nd, (rem * 10 + d) % k, tight and d == up, ns)
        return total

    return dfs(0, 0, 0, True, False)


def beautiful_brute(l, r, k):
    def ok(y):
        e = even_digit_count(y)
        return e == len(str(y)) - e and y % k == 0
    return sum(1 for y in range(l, r + 1) if ok(y))


random.seed(2827)
for _ in range(60):
    l = random.randint(1, 5000)
    r = random.randint(l, 5000)
    k = random.randint(1, 20)
    assert count_beautiful_upto(r, k) - count_beautiful_upto(l - 1, k) == beautiful_brute(l, r, k)
print("Biến thể 2827 khớp brute force. Ví dụ [10, 20], k = 3 ->", count_beautiful_upto(20, 3) - count_beautiful_upto(9, 3))

Biến thể 2827 khớp brute force. Ví dụ [10, 20], k = 3 -> 2


# 18. Bài tập vận dụng

Làm theo thứ tự, từ dễ tới khó. **Đừng xem đáp án trước** — đáp án nằm ở cuối notebook.

## Bài 1 — Đoán output (không chạy code)

Với mỗi số sau, có bao nhiêu chữ số chẵn, và nó có phải evenly good không?

```text
2024, 13579, 100, 8, 2468, 10, 1000
```

## Bài 2 — Tính tay bằng công thức khối 10

Dùng `F(x) = 5 * (x // 10) + tail`, hãy tính tay:

1. `F(57)`
2. `F(39)`
3. Số lượng số tốt trong `[40, 57]`.

Sau đó liệt kê tay các số tốt trong `[40, 57]` để kiểm tra.

## Bài 3 — Điền TODO cho Digit DP

Hoàn thiện hàm `exercise_count_upto_dp(x)` (đếm số tốt trong `[1, x]`) theo khung `dfs(pos, parity, tight, started)`. Các chỗ cần điền được đánh dấu `TODO`.

In [33]:
from functools import cache


def exercise_count_upto_dp(x: int) -> int:
    if x <= 0:
        return 0
    s = str(x)
    n = len(s)

    @cache
    def dfs(pos: int, parity: int, tight: bool, started: bool) -> int:
        if pos == n:
            # TODO 1: trả về 1 khi nào? (nhớ: số phải khác 0 và parity phải ...)
            pass
        # TODO 2: up = ... (phụ thuộc tight)
        total = 0
        # TODO 3: for d in range(...):
        #             new_started = ...
        #             new_parity = ... (chỉ lật khi new_started và d chẵn)
        #             total += dfs(pos + 1, new_parity, ..., new_started)
        return total

    raise NotImplementedError("Hãy hoàn thiện các TODO rồi xóa dòng này.")


# Tự làm xong rồi mới mở test:
# assert exercise_count_upto_dp(22) - exercise_count_upto_dp(17) == 3
# assert exercise_count_upto_dp(101) - exercise_count_upto_dp(97) == 2
# assert exercise_count_upto_dp(10) - exercise_count_upto_dp(0) == 5

## Bài 4 — Tự viết lại Cách 4

Không nhìn lại phía trên, hãy viết `exercise_count_blocks(l, r)` dùng công thức khối 10. Yêu cầu:
- Không dùng `str`.
- Xử lý đúng `q = 0`.
- Chạy được với `r = 10^15`.

In [34]:
def exercise_count_blocks(l: int, r: int) -> int:
    # TODO: viết hàm phụ F(x) = 5 * (x // 10) + tail
    # TODO: trả về F(r) - F(l - 1)
    raise NotImplementedError


# Gợi ý test:
# assert exercise_count_blocks(18, 22) == 3
# assert exercise_count_blocks(98, 101) == 2
# assert exercise_count_blocks(1, 10) == 5
# assert exercise_count_blocks(1, 10 ** 15) == 5 * 10 ** 14

## Bài 5 — Biến thể: "oddly good"

Gọi một số là **oddly good** nếu nó có **số lẻ** các chữ số chẵn (ví dụ `18`, `21`, `101`). Hãy đếm số oddly good trong `[l, r]` với `1 <= l <= r <= 10^15`.

Ví dụ: `[18, 22]` có 2 số oddly good (`18`, `21`).

In [35]:
def exercise_count_oddly_good(l: int, r: int) -> int:
    # TODO: có cần viết lại cả digit DP không? Hãy nghĩ về "phần bù".
    pass


# Gợi ý test:
# assert exercise_count_oddly_good(18, 22) == 2
# assert exercise_count_oddly_good(98, 101) == 2
# assert exercise_count_oddly_good(1, 10) == 5

## Bài 6 — Mini challenge: đúng `k` chữ số chẵn

Đếm số nguyên trong `[l, r]` có **đúng `k`** chữ số chẵn (`1 <= l <= r <= 10^15`, `0 <= k <= 16`).

Ví dụ: trong `[1, 100]` có `30` số có đúng 0 chữ số chẵn và `49` số có đúng 1 chữ số chẵn.

Lưu ý: bài này **không** còn "may mắn" như bài gốc — luật một nửa không áp dụng trực tiếp và số 0 đệm chắc chắn gây sai (xem Lỗi 4).

In [36]:
def exercise_count_exactly_k(l: int, r: int, k: int) -> int:
    # TODO: digit DP với trạng thái (pos, cnt, tight, started)
    # TODO: cắt nhánh sớm khi cnt > k
    raise NotImplementedError


# Gợi ý test:
# assert exercise_count_exactly_k(1, 100, 0) == 30
# assert exercise_count_exactly_k(1, 100, 1) == 49
# assert exercise_count_exactly_k(18, 22, 2) == 2

# 19. Gợi ý theo 3 mức

Chỉ mở mức tiếp theo khi mức trước chưa đủ.

## Bài 1
- **Mức 1 (hướng tư duy):** đếm từng chữ số thuộc `{0, 2, 4, 6, 8}`; nhớ chữ số `0`.
- **Mức 2 (cấu trúc):** `1000` có ba chữ số `0`.
- **Mức 3 (gần code):** `sum(1 for ch in str(x) if ch in "02468")`.

## Bài 2
- **Mức 1:** tách `x = 10q + t`; `5q` cho các khối đầy đủ.
- **Mức 2:** `q = 5` và `q = 3` đều có 0 chữ số chẵn → `p = 0` → cần chữ số cuối lẻ.
- **Mức 3:** `F(57) = 5*5 + (7 + 1)//2`, `F(39) = 5*3 + (9 + 1)//2`; đáp án `F(57) - F(39)`.

## Bài 3
- **Mức 1:** base case: số phải "đã bắt đầu" và có parity 0.
- **Mức 2:** `up = int(s[pos]) if tight else 9`; `new_tight = tight and d == up`.
- **Mức 3:** `new_parity = parity ^ 1 if (new_started and d % 2 == 0) else parity`.

## Bài 4
- **Mức 1:** mỗi khối 10 số có 5 số tốt; chỉ cần đếm khối dở dang.
- **Mức 2:** `q, t = divmod(x, 10)`; tính parity của `q` bằng vòng `while q > 0`.
- **Mức 3:** `tail = (t + 1) // 2 if p == 0 else t // 2 + 1`.

## Bài 5
- **Mức 1:** mỗi số hoặc evenly good, hoặc oddly good — không có lựa chọn thứ ba.
- **Mức 2:** số lượng oddly good = tổng số lượng số − số evenly good.
- **Mức 3:** `(r - l + 1) - count_evenly_good_blocks(l, r)`.

## Bài 6
- **Mức 1:** thay `parity` bằng `cnt` = số chữ số chẵn thật đã đặt.
- **Mức 2:** vẫn cần `started`; base case trả 1 khi `started and cnt == k`.
- **Mức 3:** `nc = cnt + (1 if new_started and d % 2 == 0 else 0)`; nếu `cnt > k` thì `return 0` ngay.

# Đáp án tham khảo

Chỉ xem sau khi đã tự làm. Mỗi đáp án có `assert` để tự kiểm tra.

## Đáp án Bài 1 và Bài 2

- Bài 1: `2024` → 4 (tốt), `13579` → 0 (tốt), `100` → 2 (tốt), `8` → 1 (không), `2468` → 4 (tốt), `10` → 1 (không), `1000` → 3 (không).
- Bài 2: `F(57) = 25 + 4 = 29`, `F(39) = 15 + 5 = 20`, đáp án `29 - 20 = 9`. Các số tốt: `40, 42, 44, 46, 48` (prefix `4` có parity 1 → cần chữ số cuối chẵn) và `51, 53, 55, 57` (prefix `5` parity 0 → cần chữ số cuối lẻ).

In [37]:
for x in [2024, 13579, 100, 8, 2468, 10, 1000]:
    print(f"{x:>6}: {even_digit_count(x)} chữ số chẵn -> tốt: {is_evenly_good(x)}")

assert count_upto_blocks(57) == 29
assert count_upto_blocks(39) == 20
good_40_57 = [x for x in range(40, 58) if is_evenly_good(x)]
print("Số tốt trong [40, 57]:", good_40_57, "->", len(good_40_57))
assert len(good_40_57) == 9

  2024: 4 chữ số chẵn -> tốt: True
 13579: 0 chữ số chẵn -> tốt: True
   100: 2 chữ số chẵn -> tốt: True
     8: 1 chữ số chẵn -> tốt: False
  2468: 4 chữ số chẵn -> tốt: True
    10: 1 chữ số chẵn -> tốt: False
  1000: 3 chữ số chẵn -> tốt: False
Số tốt trong [40, 57]: [40, 42, 44, 46, 48, 51, 53, 55, 57] -> 9


## Đáp án Bài 3 — Digit DP

In [38]:
from functools import cache


def answer_count_upto_dp(x: int) -> int:
    if x <= 0:
        return 0
    s = str(x)
    n = len(s)

    @cache
    def dfs(pos: int, parity: int, tight: bool, started: bool) -> int:
        if pos == n:
            return 1 if (started and parity == 0) else 0
        up = int(s[pos]) if tight else 9
        total = 0
        for d in range(up + 1):
            new_started = started or d != 0
            new_parity = parity ^ 1 if (new_started and d % 2 == 0) else parity
            total += dfs(pos + 1, new_parity, tight and d == up, new_started)
        return total

    return dfs(0, 0, True, False)


assert answer_count_upto_dp(22) - answer_count_upto_dp(17) == 3
assert answer_count_upto_dp(101) - answer_count_upto_dp(97) == 2
assert answer_count_upto_dp(10) - answer_count_upto_dp(0) == 5
print("Đáp án Bài 3 PASS")

Đáp án Bài 3 PASS


## Đáp án Bài 4 — Công thức khối 10

In [39]:
def answer_count_blocks(l: int, r: int) -> int:
    def F(x: int) -> int:
        q, t = divmod(x, 10)
        p = 0
        y = q
        while y > 0:
            y, d = divmod(y, 10)
            if d % 2 == 0:
                p ^= 1
        tail = (t + 1) // 2 if p == 0 else t // 2 + 1
        return 5 * q + tail

    return F(r) - F(l - 1)


assert answer_count_blocks(18, 22) == 3
assert answer_count_blocks(98, 101) == 2
assert answer_count_blocks(1, 10) == 5
assert answer_count_blocks(1, 10 ** 15) == 5 * 10 ** 14
print("Đáp án Bài 4 PASS")

Đáp án Bài 4 PASS


## Đáp án Bài 5 — Oddly good (phần bù)

Mỗi số nguyên dương hoặc có số chẵn, hoặc có số lẻ chữ số chẵn. Nên số oddly good = tổng số lượng − số evenly good. (Theo luật khối 10, oddly good cũng có đúng 5 số mỗi khối — trừ khối đầu `[1..9]` chỉ có 4 số: `2, 4, 6, 8`.)

In [40]:
def answer_count_oddly_good(l: int, r: int) -> int:
    return (r - l + 1) - answer_count_blocks(l, r)


assert answer_count_oddly_good(18, 22) == 2
assert answer_count_oddly_good(98, 101) == 2
assert answer_count_oddly_good(1, 10) == 5
for l, r in [(1, 500), (37, 1234), (999, 1001)]:
    brute = sum(1 for x in range(l, r + 1) if even_digit_count(x) % 2 == 1)
    assert answer_count_oddly_good(l, r) == brute
print("Đáp án Bài 5 PASS")

Đáp án Bài 5 PASS


## Đáp án Bài 6 — Đúng `k` chữ số chẵn

Trạng thái `(pos, cnt, tight, started)` với `cnt <= k <= 16` → khoảng `17 * 17 * 4` trạng thái, mỗi trạng thái 10 chữ số → vẫn rất nhanh với `r = 10^15`.

In [41]:
from functools import cache
import random


def answer_count_exactly_k(l: int, r: int, k: int) -> int:
    def count_upto(x: int) -> int:
        if x <= 0:
            return 0
        s = str(x)
        n = len(s)

        @cache
        def dfs(pos, cnt, tight, started):
            if cnt > k:
                return 0
            if pos == n:
                return 1 if (started and cnt == k) else 0
            up = int(s[pos]) if tight else 9
            total = 0
            for d in range(up + 1):
                ns = started or d != 0
                nc = cnt + (1 if (ns and d % 2 == 0) else 0)
                total += dfs(pos + 1, nc, tight and d == up, ns)
            return total

        return dfs(0, 0, True, False)

    return count_upto(r) - count_upto(l - 1)


assert answer_count_exactly_k(1, 100, 0) == 30
assert answer_count_exactly_k(1, 100, 1) == 49
assert answer_count_exactly_k(18, 22, 2) == 2

random.seed(6)
for _ in range(150):
    l = random.randint(1, 2000)
    r = random.randint(l, 2000)
    k = random.randint(0, 4)
    brute = sum(1 for x in range(l, r + 1) if even_digit_count(x) == k)
    assert answer_count_exactly_k(l, r, k) == brute
print("Đáp án Bài 6 PASS | [1, 10^15], k = 8 ->", answer_count_exactly_k(1, 10 ** 15, 8))

Đáp án Bài 6 PASS | [1, 10^15], k = 8 -> 193157171015625


# Tóm tắt kiến thức cần nhớ

## Chuỗi suy luận của bài

1. `r` tới `10^15` → không duyệt từng số; đếm **theo chữ số**.
2. Đoạn → tiền tố: `đáp án = F(r) - F(l - 1)`.
3. Chỉ cần **parity** số chữ số chẵn của prefix (2 trạng thái).
4. Digit DP `dfs(pos, parity, tight, started)` — khung tổng quát.
5. Luật một nửa: `m >= 1` chữ số tự do → đúng `10^m / 2` cách tốt (ghép cặp `d ↔ d ^ 1`).
6. Mỗi khối 10 số có 5 số tốt → `F(x) = 5 * (x // 10) + tail`.

## Template ngắn nhất (Cách 4)

```python
class Solution:
    def countEvenlyGoodIntegers(self, l: int, r: int) -> int:
        def F(x):
            q, t = divmod(x, 10)
            p, y = 0, q
            while y > 0:
                y, d = divmod(y, 10)
                if d % 2 == 0:
                    p ^= 1
            return 5 * q + ((t + 1) // 2 if p == 0 else t // 2 + 1)
        return F(r) - F(l - 1)
```

## Template tổng quát (Cách 2 — dùng lại cho bài digit DP khác)

```python
from functools import cache

def count_upto(x):
    if x <= 0:
        return 0
    s = str(x)
    @cache
    def dfs(pos, parity, tight, started):
        if pos == len(s):
            return 1 if (started and parity == 0) else 0
        up = int(s[pos]) if tight else 9
        total = 0
        for d in range(up + 1):
            ns = started or d != 0
            np = parity ^ 1 if (ns and d % 2 == 0) else parity
            total += dfs(pos + 1, np, tight and d == up, ns)
        return total
    return dfs(0, 0, True, False)
```

## Câu nhớ nhanh

> **"Đoạn thì trừ tiền tố; chữ số thì nhớ parity; còn chữ số tự do thì chia đôi."**

# Checklist trước khi submit

- [ ] Mình có đếm chữ số **`0`** là chữ số chẵn chưa? (`"02468"`, không phải `"2468"`)
- [ ] Mình có trừ **`F(l - 1)`** (không phải `F(l)`) chưa?
- [ ] `F(0)` có bằng `0` không? (số `0` không được đếm)
- [ ] Ở Cách 4, `q = 0` có cho parity `0` không (không coi `q` là chữ số `"0"`)?
- [ ] Ở digit DP, có cờ `started` (hoặc đã hiểu vì sao bài này bỏ được) chưa?
- [ ] Nếu `dfs` dùng biến ngoài `s`, mình có `cache_clear()` giữa hai lần gọi chưa (hoặc định nghĩa `dfs` bên trong `count_upto`)?
- [ ] Mọi phép chia đều là `//` hoặc `divmod`, không dùng `/`?
- [ ] Đã test 3 ví dụ của đề **và** so với brute force trên input ngẫu nhiên?
- [ ] Đã thử biên `l = r`, `l = 1`, `r = 10^15`?
- [ ] Tên method đúng `countEvenlyGoodIntegers(self, l: int, r: int) -> int`?